# TiRex-2 — DIMER zero-shot forecasting tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tirex-forecasting-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tirex-forecasting-pipeline/blob/main/tutorials/tirex_forecasting_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-NX--AI%2FTiRex--2-ffcc4d?style=flat)](https://huggingface.co/NX-AI/TiRex-2) [![Upstream](https://img.shields.io/badge/Upstream-NX--AI%2Ftirex--2-181717?style=flat&logo=github&logoColor=white)](https://github.com/NX-AI/tirex-2) [![arXiv](https://img.shields.io/badge/arXiv-2607.01204-b31b1b.svg)](https://arxiv.org/abs/2607.01204)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** zero-shot probabilistic time-series forecasting with chronological evaluation, using the pinned `NX-AI/TiRex-2` checkpoint

**This notebook is standalone.** Section 2 carries the repository's package (4 modules under `src/tirex_forecasting_pipeline/`, at revision `1df20d15d912`), the stage runner, the hash-locked requirements (57 packages) and the pinned checkpoint manifest, and verifies every carried file against its SHA-256 before use, so the notebook keeps working after export even if the repository changes or disappears. Nothing is installed into the notebook kernel: a pinned `uv` (checked by size and SHA-256) builds an isolated Python environment from the lock with `--require-hashes`, and every stage runs there in its own process, so the hosted runtime's own packages are never replaced and no restart is needed. Its only external dependencies are PyPI, the managed CPython build that `uv` downloads, and the Hugging Face Hub at the immutable revision `05e5b26db52bfb256f1ae1bdf785589850482de3` (~381 MB in all, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/3.0); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh Linux x86_64 runtime builds an isolated Python environment from the carried hash-locked requirements without touching the notebook kernel's own packages, stages and digest-verifies the pinned TiRex-2 checkpoint (~381 MB), generates a deterministic synthetic trend + season series in code, withholds the last `HORIZON` steps chronologically, validates the context into an input manifest with a recorded rejection, computes a last-value baseline and a least-squares trend + season reference on the same context, runs the zero-shot forecast on the CPU, writes an evaluation report comparing the model median with both baselines and the series' noise floor, and exports the forecast and provenance — without a repository clone, DIMER worker, credential, upload, configuration edit or runtime restart (NOTEBOOK_SPEC 2.2 §5).

**Bring Your Own Data:** Set `USE_BYOD = True` and either set `BYOD_CSV_PATH` to a CSV in this runtime (any runtime) or leave it empty in Colab to open the upload dialog, then re-run from Section 4. The CSV has a `timestamp` column and one or more numeric target columns (comma, semicolon or tab separated), with unique, increasing, regularly spaced timestamps and no empty values, and at least `MIN_CONTEXT + HORIZON` rows. A series longer than `MAX_CONTEXT + HORIZON` rows keeps its most recent rows, and the trim is reported. Every refusal names the file and the rule (a text column is named with its values). `HORIZON` is a form field, so you can roll the holdout. Uploaded files stay inside this runtime.

At inference TiRex-2 maps a context window of one or more variates (plus optional past and future-known covariates) to nine forecast quantiles (q=0.1 … q=0.9) per step of the requested horizon in a single forward pass; the pipeline reports q=0.5 as the point forecast (a model median, not a mean) and keeps all nine quantiles. **No adaptation occurs:** no gradient training, fine-tuning, streaming adaptation or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and the model configuration, and the carried package adds snapshot verification, the input contract (`validate_target`, `validate_horizon`, the covariate time-axis rules), the `mae` / `rmse` / `last_value_baseline` / `interval_coverage` helpers and the public `validate_inputs` / `evaluation_report` stages.

**Read this before Section 7: the default series is easy, so the last-value baseline flatters any forecaster.** The series is `0.02·t + sin(t/8)` plus small noise. Repeating the last value ignores both the trend and the season (MAE 0.6729), while a 4-parameter least-squares fit of trend + season on the same context reaches 0.0437 — close to the noise floor of 0.0414 that no forecaster can beat. TiRex's recorded median MAE on this sample is 0.0491. Section 7 prints all of these side by side, so you can see what the model adds and what it does not; on your own series (BYOD) the same comparison is the informative one.

**Learning objectives:** by the end of this notebook you will be able to —

1. **Make** a leakage-safe chronological holdout and **explain** why no future value may reach the model (Section 5).
2. **Read** an input manifest and a refusal message from the forecasting contract (Section 5).
3. **Interpret** a quantile forecast: the median as the point forecast, the q10–q90 band as model quantiles rather than a guaranteed interval (Section 6).
4. **Compare** the model with a last-value baseline, a least-squares trend + season reference and the noise floor, and **say** what the comparison can and cannot show (Section 7).
5. **Predict**, run and **explain** how a shorter context changes the error and the band width, in an optional activity (Section 9).

**This notebook does not demonstrate:** gradient training or fine-tuning, streaming/online adaptation, classification or regression on tabular features, anomaly detection, imputation, or calibrated prediction intervals. The quantiles are model quantiles, not guaranteed coverage; a single holdout is not a deployment-variability estimate.

## How to use this notebook

**Who this notebook is for.** Learners who can run cells in a hosted notebook and read short Python and NumPy, and who want to see a time-series foundation model used honestly: a chronological holdout, a quantile forecast read correctly, and a comparison against baselines that tells you what the model actually adds. No experience with TiRex or xLSTM is assumed; the glossary below explains every term.

**Running it.** Choose *Runtime → Run all*. A CPU runtime is enough — it is the reference path. The default path needs no edit, no upload, no account, no token and no runtime restart. Section 2 builds an isolated environment, which takes the longest.

**Where the code runs.** The notebook kernel installs nothing and imports no model library. Each learner cell calls `run_stage('…')`, which runs one stage of the carried stage runner in its own process with the isolated environment's Python, streams what it prints, and stops the notebook with the stage's own error message if it fails. Stages hand results to each other only through files.

**Two kinds of cell.** *Learner cells* (Sections 4–9) are the workflow. *Infrastructure cells* (Sections 1–3) are collapsed and titled **Infrastructure**; you may run them without studying their implementation.

**Form controls.** `USE_BYOD`, `BYOD_CSV_PATH` and `HORIZON` (Section 4); `SEASON_PERIOD` (Section 5); `RUN_ACTIVITY` and `ACTIVITY_CONTEXT` (Section 9). Leave them at their defaults for the first run.

**Section tags.** **[Concept]** — what the model does and why. **[Evaluation practice]** — how the evidence is produced and how to read it. **[Engineering]** — reproducibility, provenance and packaging.

**Predict, then check.** Before Sections 6 and 7 a **Predict before running** prompt asks you to commit to an expectation; **What to notice** follows each stage; a collapsed **Check your reasoning** answer follows each checkpoint. The sample answers quote the recorded runs of this sample.

## The task: Input → Model → Output

| Stage | Input | Model / system | Output |
|---|---|---|---|
| **Split** | a 256-step series | chronological holdout: the last `HORIZON` = 32 steps withheld | 224 context steps, 32 truth steps |
| **Validate** | the context | `validate_inputs` (shape, length 32..16,384, finiteness, covariate axis) | an input manifest with a recorded rejection |
| **Baselines** | the context | last value; least-squares level + trend + season | two reference forecasts |
| **Forecast** | the context | TiRex-2, zero-shot, on the CPU | nine quantiles per step; the median is the point forecast |
| **Evaluate** | forecast, baselines, truth | `evaluation_report` + the trend + season reference + the noise floor | MAE / RMSE, q10–q90 coverage, an interpretation |

## Roadmap

| Section | Tag | What happens | What you read |
|---|---|---|---|
| 1–3 | [Engineering] | runtime, carried code, isolated environment, verified checkpoint | versions, digests |
| 4. Data | [Evaluation practice] | synthetic series or a BYOD CSV | shape, digest, any trim |
| 5. Holdout, validation, baselines | [Evaluation practice] | split, input manifest, two baselines | the manifest, baseline errors |
| 6. Forecast | [Concept] | zero-shot quantile forecast | median vs truth, the band |
| 7. Evaluation | [Evaluation practice] | the report | the comparison and its interpretation |
| 8. Export | [Engineering] | CSV and result JSON with provenance | the files |
| 9. Optional activity | [Concept] | a shorter context, printed beside the full one | error and band width |
| Troubleshooting | [Engineering] | every failure and what to do | when something fails |
| Interpretation and conclusion | [Evaluation practice] | what was and was not shown | your conclusion |

<details>
<summary><strong>Glossary</strong> — open when a term is unfamiliar</summary>

| Term | Meaning in this notebook |
|---|---|
| **Context window** | The past observations the model sees (224 steps by default). |
| **Horizon** | How many future steps are forecast (`HORIZON`, 32 by default). |
| **Chronological holdout** | Withholding the most recent `HORIZON` steps as truth, so the forecast never sees its own future. |
| **Quantile forecast** | For each step, nine values below which the model expects the truth with probability 0.1 … 0.9. |
| **Median vs mean** | The q=0.5 quantile is the point forecast here; it is not the average of possible futures. |
| **q10–q90 band / coverage** | The range between the 0.1 and 0.9 quantiles; coverage is the share of truth points inside it (nominal 0.8, not guaranteed). |
| **Last-value baseline** | Repeat the final observed value across the horizon. |
| **Trend + season reference** | A 4-parameter least-squares fit (level, slope, one sine/cosine pair) on the context, extended forward. |
| **Noise floor** | The error of a perfect knowledge of the signal: the irreducible noise on the holdout (synthetic series only). |
| **Hash-locked environment / stage** | The isolated Python environment every stage runs in; one workflow step run as its own process. |

</details>

## Prerequisites

- **Runtime:** a fresh **Linux x86_64** runtime — Google Colab, Kaggle or a Linux Jupyter kernel. **The CPU is the reference path**: upstream CUDA execution compiles fused recurrent kernels and needs a CUDA toolkit, so the model stages hide any GPU and run on the CPU. On the CPU, `tirex-2` compiles its residual block with `torch.compile` (TorchInductor) on the first forecast, which needs a C++ compiler (`g++`) in the runtime; Colab and Kaggle images include one. The kernel's own Python version does not matter: the notebook installs nothing into it, and runs every stage with CPython 3.12.12 in an isolated environment built from 57 hash-locked packages (`tirex-2` 0.2.1, `torch` 2.8.0, `numpy` 2.3.3, `pandas` 2.3.3). About 0.4 GB of disk is needed for the checkpoint and about 8 GB for the isolated environment.
- **Knowledge:** basic Python and NumPy. The glossary below explains context window, horizon, quantile and the other forecasting terms.
- **Data:** the default sample is a deterministic synthetic trend + seasonal series (256 steps, fixed seed) generated in code, so nothing is downloaded and no private data is needed. Optional BYOD: one UTF-8 CSV with a unique `timestamp` column and one or more finite numeric target columns in chronological order, regularly spaced, at least 64 rows (32 context + 32 holdout) at the default horizon; longer than 16,416 rows keeps the most recent rows. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** `huggingface.co` (no account or token), to fetch the pinned `NX-AI/TiRex-2` checkpoint (~381 MB) at revision `05e5b26db52b…`, every file accepted only at the pinned size and SHA-256; PyPI (`pypi.org`, `files.pythonhosted.org`), for the pinned `uv` wheel and the 57 hash-locked packages; and the managed CPython build (python-build-standalone) that `uv` downloads for the isolated environment. No repository clone and no credentials are required; nothing is installed from this repository.

## 1. Check the runtime · [Engineering]

> **Infrastructure.** The code cells in Sections 1–3 are collapsed and titled **Infrastructure**. You may run them without studying their implementation; they exist for reproducibility and provenance, not as prerequisite machine-learning knowledge. The learning activities start in Section 4.

**Input:** a fresh hosted runtime. **System:** checks that it is Linux x86_64 with enough free disk, looks for a CUDA GPU, and creates a run directory under `dimer_runs/tirex_forecasting/` for the carried code, the stage logs and the hand-off state between stages. Learner-facing files are written to `outputs/`. **Output:** one dictionary naming the accelerator, the kernel's Python version and the directories. Running this cell again in the same session keeps the run directory, so the cells after it keep working; tick `NEW_RUN_DIRECTORY` for a fresh one and run Sections 2 and 3 again. The verified checkpoint is kept in `weights/` and reused.

In [ ]:
# @title Infrastructure: check the runtime, accelerator and disk; create (or keep) the run directory
NEW_RUN_DIRECTORY = False  # @param {type:"boolean"}
import base64
import hashlib
import json
import os
import platform
import shutil
import subprocess
import tempfile
import time
import uuid
from pathlib import Path

SESSION_START = time.perf_counter()
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle, or a Linux x86_64 Jupyter kernel): its locked environment is built for manylinux x86_64 wheels. A GPU is recommended, not required.')
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    accelerator = gpu.stdout.strip() if gpu.returncode == 0 else ''
except FileNotFoundError:
    accelerator = ''
if not accelerator:
    accelerator = 'none (CPU only)'
    print('No CUDA GPU detected: that is fine, the CPU is this notebook\'s reference runtime.')
STEM = 'tirex_forecasting'
# Re-running this cell keeps the run directory of this session, so the cells after it keep working; tick
# NEW_RUN_DIRECTORY (then run the cells below again) for a fresh one.
if NEW_RUN_DIRECTORY or not isinstance(globals().get('ROOT'), Path) or not globals()['ROOT'].is_dir():
    ROOT = Path.cwd() / 'dimer_runs' / STEM / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUTS = Path.cwd() / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
WEIGHTS = Path.cwd() / 'weights'
WEIGHTS.mkdir(exist_ok=True)
ENV_ROOT = Path(tempfile.gettempdir()) / 'dimer_uv'
staged_gib = sum(p.stat().st_size for p in WEIGHTS.rglob('*') if p.is_file()) / 1024**3
need = {'weights': max(0.0, 0.4 - staged_gib), 'environment': 8.0}
free = {'weights': shutil.disk_usage(WEIGHTS).free / 1024**3, 'environment': shutil.disk_usage(tempfile.gettempdir()).free / 1024**3}
if os.stat(WEIGHTS).st_dev == os.stat(tempfile.gettempdir()).st_dev:
    short = free['weights'] < need['weights'] + need['environment']
else:
    short = free['weights'] < need['weights'] or free['environment'] < need['environment']
if short:
    raise RuntimeError(f'Not enough free disk: need about {need} GiB, free {free} GiB. Start a fresh runtime (see Troubleshooting).')
print({'accelerator': accelerator, 'kernel_python': platform.python_version(), 'run_directory': str(ROOT), 'outputs': str(OUTPUTS), 'weights': str(WEIGHTS), 'environment_root': str(ENV_ROOT), 'free_gib': {k: round(v, 1) for k, v in free.items()}})

{'accelerator': 'Tesla T4, 15360 MiB', 'kernel_python': '3.13.15', 'run_directory': '/content/dimer_runs/tirex_forecasting/7bd860dc1dce', 'outputs': '/content/outputs', 'weights': '/content/weights', 'environment_root': '/tmp/dimer_uv', 'free_gib': {'weights': 193.3, 'environment': 193.3}}


## 2. Carry the code and build the isolated environment · [Engineering]

> **Infrastructure.** The next two code cells are collapsed. The first **is** the repository's code, carried so that this notebook works on its own; the second builds the environment every stage runs in.

The first cell holds, as text, the files the workflow needs: the package's 4 modules under `src/tirex_forecasting_pipeline/` (at revision `1df20d15d912`), the stage runner `tutorial_stages.py`, the hash-locked `requirements.txt` (57 packages), the pinned checkpoint manifest and the licence. It writes each file into the run directory and checks its SHA-256 against `CARRIED_HASHES`, stopping on any mismatch. The text is the repository's files byte for byte; the repository's parity tests fail whenever the two diverge, so what runs here is what the repository tests. Nothing in this cell runs a model.

**Expected result:** `carried_files`, `verified: True`, and the repository revision the notebook was generated from.

In [ ]:
# @title Infrastructure: write and verify the carried package, stage runner, lock, manifest and samples
CARRIED_FILES = {'src/tirex_forecasting_pipeline/__init__.py': 'from .evaluation import interval_coverage, last_value_baseline, mae, rmse\nfrom .pipeline import (\n    DEFAULT_WEIGHTS_DIR,\n    INPUT_SCHEMA,\n    MODEL_ID,\n    MODEL_KEY,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    QUANTILES,\n    TiRexForecastPipeline,\n    evaluation_report,\n    stage_missing_files,\n    validate_inputs,\n    verify_snapshot,\n)\nfrom .validation import MAX_CONTEXT, MAX_HORIZON, MIN_CONTEXT, validate_horizon, validate_target\n\n__all__ = [\n    "DEFAULT_WEIGHTS_DIR",\n    "INPUT_SCHEMA",\n    "MAX_CONTEXT",\n    "MAX_HORIZON",\n    "MIN_CONTEXT",\n    "MODEL_ID",\n    "MODEL_KEY",\n    "MODEL_LICENSE",\n    "MODEL_REVISION",\n    "QUANTILES",\n    "TiRexForecastPipeline",\n    "evaluation_report",\n    "interval_coverage",\n    "last_value_baseline",\n    "mae",\n    "rmse",\n    "stage_missing_files",\n    "validate_horizon",\n    "validate_inputs",\n    "validate_target",\n    "verify_snapshot",\n]\n', 'src/tirex_forecasting_pipeline/evaluation.py': 'from __future__ import annotations\n\nimport numpy as np\n\n\ndef mae(y_true, y_pred) -> float:\n    actual = np.asarray(y_true, dtype=float)\n    predicted = np.asarray(y_pred, dtype=float)\n    if actual.shape != predicted.shape:\n        raise ValueError("y_true and y_pred must have the same shape")\n    return float(np.mean(np.abs(actual - predicted)))\n\n\ndef rmse(y_true, y_pred) -> float:\n    actual = np.asarray(y_true, dtype=float)\n    predicted = np.asarray(y_pred, dtype=float)\n    if actual.shape != predicted.shape:\n        raise ValueError("y_true and y_pred must have the same shape")\n    return float(np.sqrt(np.mean((actual - predicted) ** 2)))\n\n\ndef last_value_baseline(context, horizon: int) -> np.ndarray:\n    values = np.asarray(context, dtype=float)\n    if values.ndim == 1:\n        values = values[None, :]\n    if values.ndim != 2 or values.shape[1] < 1:\n        raise ValueError(\n            "context must have shape (variates, time) with at least one observation"\n        )\n    return np.repeat(values[:, -1:], horizon, axis=1)\n\n\ndef interval_coverage(y_true, lower, upper) -> float:\n    actual = np.asarray(y_true, dtype=float)\n    lower_bound = np.asarray(lower, dtype=float)\n    upper_bound = np.asarray(upper, dtype=float)\n    if not (actual.shape == lower_bound.shape == upper_bound.shape):\n        raise ValueError("coverage arrays must share shape")\n    return float(np.mean((actual >= lower_bound) & (actual <= upper_bound)))\n', 'src/tirex_forecasting_pipeline/pipeline.py': '"""Zero-shot probabilistic forecasting with the pinned ``NX-AI/TiRex-2`` checkpoint.\n\nThe class loads the checkpoint only from a digest-verified local snapshot (``weights/<key>/``:\n``model-config.yaml`` + ``model.ckpt``) or, when explicitly allowed, from the Hugging Face Hub at\nthe pinned revision. Model construction and checkpoint deserialisation happen inside the upstream\n``tirex2`` package (``weights_only`` PyTorch state-dict loading); this module never executes\nmodel-repository code.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nfrom .evaluation import interval_coverage, last_value_baseline, mae, rmse\nfrom .validation import MAX_CONTEXT, MAX_HORIZON, MIN_CONTEXT, validate_horizon, validate_target\n\nMODEL_ID = "NX-AI/TiRex-2"\nMODEL_REVISION = "05e5b26db52bfb256f1ae1bdf785589850482de3"\nMODEL_LICENSE = "Apache-2.0"\nMODEL_KEY = "tirex-2"\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\nMANIFEST_NAME = "dimer-base-manifest.json"\nCONFIG_FILE = "model-config.yaml"\nWEIGHTS_FILE = "model.ckpt"\nQUANTILES = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)\nMEDIAN_INDEX = 4  # QUANTILES[4] == 0.5: the point forecast is the model median, not a mean\nPOINT_FORECAST = "median (q=0.5)"\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for chunk in iter(lambda: fh.read(1 << 20), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    """Check a local snapshot against its manifest; raise naming the first mismatch."""\n    root = Path(path or DEFAULT_WEIGHTS_DIR)\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")\n    with open(manifest_path, encoding="utf-8") as fh:\n        manifest = json.load(fh)\n    if manifest.get("modelId") != MODEL_ID:\n        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n    if manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n    for entry in manifest.get("files", []):\n        file_path = root / entry["path"]\n        if not file_path.is_file():\n            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n        size = file_path.stat().st_size\n        if size != entry["bytes"]:\n            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n        digest = _sha256(file_path)\n        if digest != entry["sha256"]:\n            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n    return {"path": str(root), **manifest}\n\n\ndef _hub_download(relative_path: str, root: Path) -> None:\n    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n    from huggingface_hub import hf_hub_download\n\n    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n    git-ignores the checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n    with open(manifest_path, encoding="utf-8") as fh:\n        manifest = json.load(fh)\n    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(\n            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n        )\n    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f"snapshot at {root} is missing {missing}; "\n            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n        )\n    fetch = downloader or _hub_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\ndef _validate_covariates(value, *, expected_length: int, name: str) -> np.ndarray | None:\n    if value is None:\n        return None\n    array = np.asarray(value, dtype=np.float32)\n    if array.ndim == 1:\n        array = array[None, :]\n    if array.ndim != 2:\n        raise ValueError(f"{name} must be 1D or 2D with shape (covariates, time)")\n    if array.shape[0] < 1:\n        raise ValueError(f"{name} must contain at least one covariate")\n    if array.shape[1] != expected_length:\n        raise ValueError(\n            f"{name} must contain exactly {expected_length} time steps; got {array.shape[1]}"\n        )\n    if not np.isfinite(array).all():\n        raise ValueError(f"{name} must contain only finite values")\n    return array\n\n\nINPUT_SCHEMA: dict[str, Any] = {\n    "input": "1D array (time) or 2D array (variates, time) of finite numbers, cast to float32",\n    "context_length": [MIN_CONTEXT, MAX_CONTEXT],\n    "horizon": [1, MAX_HORIZON],\n    "past_covariates": "optional (covariates, context_length), finite",\n    "future_covariates": "optional (covariates, context_length + horizon), finite",\n    "preprocessing": "none in this module; upstream tirex2 scales and patch-tokenises the context internally",\n}\n\n\ndef _check_inputs(\n    target: Any, horizon: int, past_covariates: Any = None, future_covariates: Any = None\n) -> tuple[np.ndarray, np.ndarray | None, np.ndarray | None]:\n    """Raise ValueError naming the first violated ceiling; return the validated arrays."""\n    values = validate_target(target)\n    validate_horizon(horizon)\n    context_length = values.shape[1]\n    past_values = _validate_covariates(\n        past_covariates, expected_length=context_length, name="past_covariates"\n    )\n    future_values = _validate_covariates(\n        future_covariates, expected_length=context_length + horizon, name="future_covariates"\n    )\n    return values, past_values, future_values\n\n\ndef validate_inputs(\n    target: Any,\n    *,\n    horizon: int,\n    past_covariates: Any = None,\n    future_covariates: Any = None,\n    names: Sequence[str] | None = None,\n) -> dict[str, Any]:\n    """Validation stage: return the input manifest (schema, per-variate observations, verdict).\n\n    Rejection is reported by raising exactly as ``forecast`` would; a caller that wants the\n    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.\n    """\n    values, past_values, future_values = _check_inputs(\n        target, horizon, past_covariates, future_covariates\n    )\n    if names is not None and len(names) != values.shape[0]:\n        raise ValueError("names must have one entry per variate")\n    return {\n        "schema": dict(INPUT_SCHEMA),\n        "inputs": [\n            {\n                "id": names[i] if names else f"variate-{i}",\n                "context_length": int(values.shape[1]),\n                "min": float(values[i].min()),\n                "max": float(values[i].max()),\n            }\n            for i in range(values.shape[0])\n        ],\n        "horizon": horizon,\n        "past_covariates": int(past_values.shape[0]) if past_values is not None else 0,\n        "future_covariates": int(future_values.shape[0]) if future_values is not None else 0,\n        "verdict": "accepted",\n        "findings": [],\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n\n\ndef evaluation_report(\n    result: Mapping[str, Any],\n    truth: Any = None,\n    *,\n    context: Any = None,\n    sample_kind: str = "synthetic",\n) -> dict[str, Any]:\n    """Evaluation stage: a machine-readable report even when nothing is measurable.\n\n    With ``truth`` (the withheld future, shape ``(variates, horizon)``) the report carries the\n    repository\'s ``mae`` / ``rmse`` on the median forecast and, when ``context`` is supplied, the\n    same metrics for ``last_value_baseline`` plus ``interval_coverage`` of the q10–q90 band, with\n    verdict ``sample-sanity``. Without ``truth`` the verdict is ``not-measurable``.\n    """\n    median = np.asarray(result["median"], dtype=float)\n    base = {\n        "task": "zero-shot probabilistic time-series forecasting",\n        "point_forecast": POINT_FORECAST,\n        "score_semantics": (\n            "quantile levels 0.1..0.9 are model quantiles, not calibrated confidence intervals"\n        ),\n        "sample_kind": sample_kind,\n        "horizon": int(result["horizon"]),\n        "context_length": int(result["context_length"]),\n        "n_variates": int(median.shape[0]),\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n    if truth is None:\n        return {\n            **base,\n            "metrics": [],\n            "baselines": [],\n            "verdict": "not-measurable",\n            "reason": "no withheld future values were supplied for the forecast horizon",\n            "needs": (\n                "a chronological holdout: withhold the final `horizon` observations of the target, "\n                "forecast from the remaining context, and score the median with mae/rmse against them "\n                "and against the "\n                "last_value_baseline, repeated over representative periods"\n            ),\n        }\n    actual = np.asarray(truth, dtype=float)\n    if actual.ndim == 1:\n        actual = actual[None, :]\n    if actual.shape != median.shape:\n        raise ValueError(f"truth shape {actual.shape} != median shape {median.shape}")\n    estimation = "single chronological holdout, no dispersion estimate"\n    metrics = [\n        {"id": "mae", "value": mae(actual, median), "estimation": estimation},\n        {"id": "rmse", "value": rmse(actual, median), "estimation": estimation},\n    ]\n    baselines = []\n    quantiles = np.asarray(result["quantiles"], dtype=float)\n    levels = list(result["quantile_levels"])\n    if quantiles.shape[:1] == actual.shape[:1] and 0.1 in levels and 0.9 in levels:\n        metrics.append(\n            {\n                "id": "interval_coverage",\n                "band": [0.1, 0.9],\n                "value": interval_coverage(\n                    actual, quantiles[:, levels.index(0.1), :], quantiles[:, levels.index(0.9), :]\n                ),\n                "nominal": 0.8,\n                "estimation": estimation,\n            }\n        )\n    if context is not None:\n        baseline = last_value_baseline(context, int(result["horizon"]))\n        baselines.append(\n            {\n                "id": "last_value_baseline",\n                "metrics": [\n                    {"id": "mae", "value": mae(actual, baseline)},\n                    {"id": "rmse", "value": rmse(actual, baseline)},\n                ],\n            }\n        )\n    return {\n        **base,\n        "metrics": metrics,\n        "baselines": baselines,\n        "verdict": "sample-sanity",\n        "reason": (\n            f"one chronological holdout of {actual.shape[1]} step(s) on the tutorial sample; not a benchmark"\n        ),\n        "needs": (\n            "repeated holdouts over representative periods of the deployment series for any "\n            "generalisable claim"\n        ),\n    }\n\n\n@dataclass\nclass TiRexForecastPipeline:\n    _model: Any\n    device: str\n    source: str = "injected"\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        device: str = "cpu",\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> TiRexForecastPipeline:\n        import torch\n\n        if device.startswith("cpu") and torch.cuda.is_available():\n            # Upstream xlstm resolves CUDA include paths at import time whenever a GPU is\n            # visible, even though CPU inference compiles no CUDA kernel (tirex-2\'s torch.compile of the\n            # residual block does compile CPU code with TorchInductor, which needs a C++ compiler).\n            # Use torch\'s own resolution (CUDA_HOME, CUDA_PATH, nvcc on PATH, /usr/local/cuda)\n            # as the oracle and surface a typed error instead of letting an OSError escape from inside\n            # the import.\n            from torch.utils.cpp_extension import CUDA_HOME\n\n            if CUDA_HOME is None:\n                raise RuntimeError(\n                    "A CUDA device is visible but no CUDA toolkit was found (CUDA_HOME/CUDA_PATH "\n                    "unset, no nvcc on PATH, no /usr/local/cuda); upstream xlstm needs the toolkit "\n                    "at import even for device=\'cpu\'. Hide the GPU with CUDA_VISIBLE_DEVICES=\'\' "\n                    "before importing torch, or set CUDA_HOME."\n                )\n\n        from tirex2 import load_model\n\n        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)\n        if (root / MANIFEST_NAME).is_file():\n            stage_missing_files(root, allow_download=allow_download)\n            verify_snapshot(root)\n            # A directory argument makes tirex2 read model-config.yaml + model.ckpt from it directly\n            # (no Hub resolution, no cache lookup).\n            model = load_model(str(root), device=device)\n            source = "local-snapshot"\n        elif allow_download:\n            model = load_model(\n                MODEL_ID,\n                device=device,\n                hf_kwargs={"revision": MODEL_REVISION},\n            )\n            source = "hf-hub"\n        else:\n            raise FileNotFoundError(\n                f"no verified snapshot at {root} and allow_download=False; "\n                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"\n            )\n        return cls(model, device, source)\n\n    def forecast(\n        self,\n        target,\n        *,\n        horizon: int,\n        past_covariates=None,\n        future_covariates=None,\n    ) -> dict[str, Any]:\n        values, past_values, future_values = _check_inputs(\n            target, horizon, past_covariates, future_covariates\n        )\n        context_length = values.shape[1]\n\n        import torch\n        from tirex2 import TimeseriesType\n\n        timeseries = TimeseriesType(\n            target=torch.from_numpy(values),\n            past_covariates=(\n                torch.from_numpy(past_values) if past_values is not None else None\n            ),\n            future_covariates=(\n                torch.from_numpy(future_values) if future_values is not None else None\n            ),\n        )\n        quantiles = np.asarray(\n            self._model.forecast(\n                [timeseries],\n                prediction_length=horizon,\n                output_type="numpy",\n            )[0],\n            dtype=float,\n        )\n        expected_shape = (values.shape[0], len(QUANTILES), horizon)\n        if quantiles.shape != expected_shape:\n            raise RuntimeError(\n                f"unexpected TiRex forecast shape: {quantiles.shape}; "\n                f"expected {expected_shape}"\n            )\n        return {\n            "quantiles": quantiles,\n            "quantile_levels": QUANTILES,\n            "median": quantiles[:, MEDIAN_INDEX, :],\n            "point_forecast": POINT_FORECAST,\n            "model_id": MODEL_ID,\n            "model_revision": MODEL_REVISION,\n            "horizon": horizon,\n            "context_length": context_length,\n            "n_variates": values.shape[0],\n            "past_covariates": past_values.shape[0] if past_values is not None else 0,\n            "future_covariates": future_values.shape[0] if future_values is not None else 0,\n            "device": self.device,\n            "source": self.source,\n        }\n', 'src/tirex_forecasting_pipeline/validation.py': 'from __future__ import annotations\n\nimport numpy as np\n\nMIN_CONTEXT = 32  # observations; shorter targets are rejected before any model work\nMAX_CONTEXT = 16_384  # observations per variate\nMAX_HORIZON = 4096  # forecast steps per call\n\n\ndef validate_target(target, *, min_context: int = MIN_CONTEXT, max_context: int = MAX_CONTEXT) -> np.ndarray:\n    values = np.asarray(target, dtype=np.float32)\n    if values.ndim == 1:\n        values = values[None, :]\n    if values.ndim != 2:\n        raise ValueError("target must be 1D or 2D with shape (variates, time)")\n    if not min_context <= values.shape[1] <= max_context:\n        raise ValueError(\n            f"context length must be between {min_context} and {max_context}"\n        )\n    if not np.isfinite(values).all():\n        raise ValueError("target must contain only finite values")\n    return values\n\n\ndef validate_horizon(horizon: int, *, max_horizon: int = MAX_HORIZON) -> int:\n    if not isinstance(horizon, int) or not 1 <= horizon <= max_horizon:\n        raise ValueError(f"horizon must be an integer from 1 to {max_horizon}")\n    return horizon\n', 'tutorial_stages.py': '"""Stage runner for the standalone TiRex-2 zero-shot forecasting tutorial (NOTEBOOK_SPEC 2.2 §25.13 isolated environment).\n\nThe tutorial notebook carries this file verbatim (as ``tutorial_stages.py`` in its run directory, beside the carried\npackage under ``src/``) and runs every stage with the interpreter of an isolated, hash-locked environment::\n\n    python -u tutorial_stages.py --root RUN_DIR --outputs OUTPUTS --weights WEIGHTS --stage data --options \'{...}\'\n\nNothing is installed into the notebook kernel. Each stage is a separate process and starts from files only: the\nverified snapshot under ``--weights``, the series and records of earlier stages under ``RUN_DIR/state`` (NumPy ``.npy``\nand JSON), and the learner-facing exports under ``--outputs``. The model runs on the CPU, the reference path: model\nstages hide any GPU (``CUDA_VISIBLE_DEVICES=\'\'``) before importing torch, because upstream xlstm otherwise needs a CUDA\ntoolkit at import even for CPU inference. On failure a stage writes ``RUN_DIR/state/<stage>.error.json``, which the\nnotebook re-raises in the kernel.\n\nStages: weights → data → validate → forecast → evaluate → export, plus the optional ``activity``. ``data`` and\n``validate`` import no model library, so CI exercises them directly. No stage asserts a quality level.\n"""\n# ruff: noqa: E501  -- the printed dictionaries are the learner-facing output; they are kept on one line each\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib\nimport json\nimport math\nimport os\nimport shutil\nimport sys\nimport traceback\nfrom pathlib import Path\nfrom typing import Any\n\nSTEM = "tirex_forecasting"\nPACKAGE = "tirex_forecasting_pipeline"\nSYNTHETIC_STEPS = 256\nSYNTHETIC_SEED = 7\nSYNTHETIC_PERIOD = 2 * math.pi * 8  # sin(t / 8): the generator\'s own season, in steps\nNOISE_SD = 0.05\nDELIMITERS = (",", ";", "\\t")\n\n\nclass Run:\n    """Paths of one run: carried sources and state under ``root``; learner-facing files under ``outputs``."""\n\n    def __init__(self, root: Path, outputs: Path, weights: Path, options: dict[str, Any]) -> None:\n        self.root = root\n        self.out = outputs\n        self.weights = weights\n        self.options = options\n        self.state = root / "state"\n        self.out.mkdir(parents=True, exist_ok=True)\n        self.state.mkdir(parents=True, exist_ok=True)\n\n    def write_state(self, name: str, value: Any) -> Path:\n        path = self.state / name\n        path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n        return path\n\n    def read_state(self, name: str, needed_by: str) -> Any:\n        path = self.state / name\n        if not path.is_file():\n            raise RuntimeError(f"{name} is missing: run the stage that writes it before \'{needed_by}\' (run the notebook in order from Section 4)")\n        return json.loads(path.read_text(encoding="utf-8"))\n\n    def write_output(self, name: str, value: Any) -> Path:\n        path = self.out / name\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n        return path\n\n\ndef package(root: Path):\n    src = str(root / "src")\n    if src not in sys.path:\n        sys.path.insert(0, src)\n    return importlib.import_module(PACKAGE)\n\n\ndef pipeline_module(root: Path):\n    package(root)\n    return importlib.import_module(f"{PACKAGE}.pipeline")\n\n\ndef snapshot_dir(run: Run, P) -> Path:\n    return run.weights / P.MODEL_KEY\n\n\ndef rounded(value: Any, digits: int = 4) -> Any:\n    if isinstance(value, float):\n        return round(value, digits) if math.isfinite(value) else value\n    if isinstance(value, dict):\n        return {k: rounded(v, digits) for k, v in value.items()}\n    if isinstance(value, list | tuple):\n        return [rounded(v, digits) for v in value]\n    return value\n\n\ndef load_series(run: Run, needed_by: str):\n    import numpy as np\n\n    data = run.read_state("data.json", needed_by)\n    return data, np.load(run.state / "series.npy")\n\n\n# --------------------------------------------------------------------------------------------------\n# weights\n# --------------------------------------------------------------------------------------------------\n\n\ndef stage_weights(run: Run) -> None:\n    P = package(run.root)\n    M = pipeline_module(run.root)\n    snapshot = snapshot_dir(run, P)\n    snapshot.mkdir(parents=True, exist_ok=True)\n    carried = run.root / "weights" / P.MODEL_KEY / M.MANIFEST_NAME\n    manifest = json.loads(carried.read_text(encoding="utf-8"))\n    if (manifest["modelId"], manifest["revision"]) != (P.MODEL_ID, P.MODEL_REVISION):\n        raise RuntimeError("the carried manifest does not name the identity carried by the package; regenerate the notebook")\n    shutil.copyfile(carried, snapshot / M.MANIFEST_NAME)\n    print({"model_id": P.MODEL_ID, "revision": P.MODEL_REVISION, "license": P.MODEL_LICENSE, "files": len(manifest["files"]), "total_bytes": manifest["totalBytes"]})\n    fetched = P.stage_missing_files(snapshot, allow_download=True)\n    print({"weights_dir": str(snapshot), "fetched": fetched})\n    verified = P.verify_snapshot(snapshot)\n    print({"verified": verified, "checkpoint": M.WEIGHTS_FILE, "note": "model.ckpt is a PyTorch checkpoint (a pickle container); tirex2 loads it with weights_only, and it is accepted only at the pinned size and SHA-256"})\n    run.write_state("weights.json", {"snapshot": str(snapshot), "fetched": fetched})\n\n\n# --------------------------------------------------------------------------------------------------\n# data (TRX-m2) and validation with baselines (TRX-m1); no model library\n# --------------------------------------------------------------------------------------------------\n\n\ndef synthetic_series():\n    import numpy as np\n\n    rng = np.random.default_rng(SYNTHETIC_SEED)\n    t = np.arange(SYNTHETIC_STEPS)\n    signal = 0.02 * t + np.sin(t / 8)\n    series = (signal + rng.normal(0, NOISE_SD, len(t)))[None, :]\n    return series, signal\n\n\ndef sniff_delimiter(path: Path) -> str:\n    with path.open(newline="", encoding="utf-8-sig") as handle:\n        first = handle.readline()\n    counts = {d: first.count(d) for d in DELIMITERS}\n    best = max(counts, key=counts.get)\n    return best if counts[best] else ","\n\n\ndef read_byod_csv(P, path: Path, horizon: int) -> dict[str, Any]:\n    """Read a BYOD CSV with every refusal naming the file and the rule; trim a long series to the most recent rows."""\n    import numpy as np\n    import pandas as pd\n\n    if not path.is_file():\n        raise FileNotFoundError(f"BYOD_CSV_PATH {str(path)!r} is not a file in this runtime: upload the CSV or correct the path.")\n    delimiter = sniff_delimiter(path)\n    with path.open(newline="", encoding="utf-8-sig") as handle:\n        header = next(csv.reader(handle, delimiter=delimiter), [])\n    header = [h.strip() for h in header]\n    if not header or any(not h for h in header) or len(header) != len(set(header)):\n        raise ValueError(f"{path.name}: the header must have non-empty, unique column names (duplicates are refused before pandas could rename them); got {header}")\n    if "timestamp" not in header:\n        raise ValueError(f"{path.name}: no `timestamp` column in the header {header} (delimiter read as {delimiter!r}; comma, semicolon and tab are recognised)")\n    names = [c for c in header if c != "timestamp"]\n    if not names:\n        raise ValueError(f"{path.name}: no target column besides `timestamp`")\n    frame = pd.read_csv(path, sep=delimiter, encoding="utf-8-sig", dtype=str, keep_default_na=False)\n    frame.columns = [c.strip() for c in frame.columns]\n    bad = {}\n    missing = {}\n    for column in names:\n        raw = frame[column].str.strip()\n        numbers = pd.to_numeric(raw.where(raw != ""), errors="coerce")\n        text = raw[(raw != "") & numbers.isna()]\n        if len(text):\n            bad[column] = text.unique().tolist()[:3]\n        if (raw == "").any():\n            missing[column] = [int(i) + 2 for i in raw.index[raw == ""][:3]]  # +2: header line, 1-based\n    if bad:\n        raise ValueError(f"{path.name}: target columns must be numeric; non-numeric values in {bad}. Remove text columns (or move identifiers out of the file) and fix the values.")\n    if missing:\n        raise ValueError(f"{path.name}: empty values in {missing} (CSV line numbers); this pipeline does not impute — fill or drop those rows so the series stays regularly spaced.")\n    try:\n        timestamps = pd.to_datetime(frame["timestamp"].str.strip(), errors="raise")\n    except (ValueError, TypeError) as exc:\n        raise ValueError(f"{path.name}: the `timestamp` column does not parse as dates/times ({exc})") from None\n    if timestamps.duplicated().any() or not timestamps.is_monotonic_increasing:\n        raise ValueError(f"{path.name}: timestamps must be unique and strictly increasing")\n    steps = timestamps.diff().dropna()\n    if len(steps) and steps.nunique() != 1:\n        raise ValueError(f"{path.name}: timestamps must be regularly spaced (found {steps.nunique()} different gaps, e.g. {sorted(map(str, steps.unique()))[:3]}); fill or resample the gaps first")\n    series = frame[names].astype(float).to_numpy().T\n    keep = P.MAX_CONTEXT + horizon\n    trimmed = None\n    if series.shape[1] > keep:\n        trimmed = {"original_rows": int(series.shape[1]), "kept_rows": keep, "dropped_oldest": int(series.shape[1] - keep), "first_kept_timestamp": str(timestamps.iloc[-keep])}\n        series = series[:, -keep:]\n        timestamps = timestamps.iloc[-keep:]\n    if series.shape[1] < P.MIN_CONTEXT + horizon:\n        raise ValueError(f"{path.name}: {series.shape[1]} rows; this tutorial needs at least {P.MIN_CONTEXT + horizon} ({P.MIN_CONTEXT} context + {horizon} holdout)")\n    values = P.validate_target(series, max_context=keep)\n    return {"series": np.asarray(values, dtype=float), "names": names, "delimiter": delimiter, "trimmed": trimmed, "first": str(timestamps.iloc[0]), "last": str(timestamps.iloc[-1]), "step": str(steps.iloc[0]) if len(steps) else None}\n\n\ndef stage_data(run: Run) -> None:\n    import numpy as np\n\n    P = package(run.root)\n    opts = run.options\n    use_byod = bool(opts.get("use_byod", False))\n    horizon = P.validate_horizon(int(opts.get("horizon", 32)))\n    for path in sorted(run.out.glob(f"{STEM}_*")):\n        if path.is_file():\n            path.unlink()\n    for name in ("data.json", "validated.json", "forecast.json", "evaluate.json"):\n        (run.state / name).unlink(missing_ok=True)\n    if use_byod:\n        path = Path(opts.get("byod_csv_path") or "")\n        info = read_byod_csv(P, path, horizon)\n        series, names = info["series"], info["names"]\n        record = {"sample_kind": "BYOD", "name": path.name, "delimiter": info["delimiter"], "trimmed": info["trimmed"], "time_range": [info["first"], info["last"]], "step": info["step"]}\n        if info["trimmed"]:\n            print({"trimmed": info["trimmed"], "note": f"kept the most recent MAX_CONTEXT + HORIZON = {info[\'trimmed\'][\'kept_rows\']} rows; older rows cannot be used as context"})\n    else:\n        series, _signal = synthetic_series()\n        names = ["synthetic-trend-season"]\n        record = {"sample_kind": "synthetic", "name": "synthetic_trend_season_256.csv", "generator": "0.02·t + sin(t/8) + N(0, 0.05), seed 7"}\n    if series.shape[1] < P.MIN_CONTEXT + horizon:\n        raise ValueError(f"this tutorial needs at least {P.MIN_CONTEXT + horizon} rows ({P.MIN_CONTEXT} context + {horizon} holdout); got {series.shape[1]}")\n    digest = hashlib.sha256(np.ascontiguousarray(series, dtype=np.float32).tobytes()).hexdigest()\n    np.save(run.state / "series.npy", series)\n    run.write_state("data.json", {**record, "names": names, "horizon": horizon, "shape": list(series.shape), "float32_sha256": digest})\n    print({**{k: v for k, v in record.items() if k != "trimmed"}, "shape": list(series.shape), "horizon": horizon, "float32_sha256": digest})\n\n\ndef estimate_period(context) -> tuple[float | None, str]:\n    """The dominant period of the linearly detrended context: the largest periodogram peak (3 steps up to half the context),\n    refined by the least-squares fit between the neighbouring frequency bins (the FFT grid alone drifts in phase over a\n    long context)."""\n    import numpy as np\n\n    y = np.asarray(context, dtype=float)\n    n = len(y)\n    t = np.arange(n, dtype=float)\n    X = np.stack([np.ones_like(t), t], 1)\n    resid = y - X @ np.linalg.lstsq(X, y, rcond=None)[0]\n    power = np.abs(np.fft.rfft(resid)) ** 2\n    k_min = 2  # at most half the context per period\n    k_max = n // 3  # at least 3 steps per period\n    if k_max < k_min:\n        return None, "context too short to estimate a season"\n    k = int(np.argmax(power[k_min : k_max + 1])) + k_min\n\n    def sse(period: float) -> float:\n        D = np.stack([np.ones_like(t), t, np.sin(2 * np.pi * t / period), np.cos(2 * np.pi * t / period)], 1)\n        r = y - D @ np.linalg.lstsq(D, y, rcond=None)[0]\n        return float(r @ r)\n\n    candidates = np.linspace(n / (k + 1), n / max(k - 1, 1), 201)\n    best = float(min(candidates, key=sse))\n    return best, f"largest periodogram peak of the detrended context (frequency {k}/{n}), refined by least squares"\n\n\ndef least_squares_forecast(context, horizon: int, period: float | None):\n    """Fit level + trend (+ one sine/cosine pair at ``period``) to each variate\'s context by least squares and extend it."""\n    import numpy as np\n\n    values = np.atleast_2d(np.asarray(context, dtype=float))\n    n = values.shape[1]\n    t_fit = np.arange(n, dtype=float)\n    t_new = np.arange(n, n + horizon, dtype=float)\n\n    def design(t):\n        columns = [np.ones_like(t), t]\n        if period:\n            columns += [np.sin(2 * np.pi * t / period), np.cos(2 * np.pi * t / period)]\n        return np.stack(columns, 1)\n\n    out = np.empty((values.shape[0], horizon))\n    for i, y in enumerate(values):\n        coef = np.linalg.lstsq(design(t_fit), y, rcond=None)[0]\n        out[i] = design(t_new) @ coef\n    return out\n\n\ndef stage_validate(run: Run) -> None:\n    import numpy as np\n\n    P = package(run.root)\n    data, series = load_series(run, "validate")\n    horizon = data["horizon"]\n    context, truth = series[:, :-horizon], series[:, -horizon:]\n    print({"ceilings": {"MIN_CONTEXT": P.MIN_CONTEXT, "MAX_CONTEXT": P.MAX_CONTEXT, "MAX_HORIZON": P.MAX_HORIZON}})\n    manifest = P.validate_inputs(context, horizon=horizon, names=data["names"])\n    try:\n        P.validate_inputs(np.ones(P.MIN_CONTEXT - 1), horizon=horizon)\n    except ValueError as exc:\n        manifest["findings"].append({"input": "short-context-probe", "verdict": "rejected", "message": str(exc)})\n    if data.get("trimmed"):\n        manifest["findings"].append({"input": data["name"], "verdict": "trimmed", "message": f"kept the most recent {data[\'trimmed\'][\'kept_rows\']} of {data[\'trimmed\'][\'original_rows\']} rows"})\n    run.write_output(f"{STEM}_input_manifest.json", manifest)\n    print(json.dumps(manifest, indent=2))\n    season = float(run.options.get("season_period") or 0)\n    if data["sample_kind"] == "synthetic":\n        period, period_source = SYNTHETIC_PERIOD, "the generator\'s own season (sin(t/8), 2π·8 ≈ 50.27 steps): the right functional family, which is the point of the reference"\n    elif season > 0:\n        period, period_source = season, "SEASON_PERIOD set in Section 5"\n    else:\n        estimates = [estimate_period(row) for row in context]\n        period, period_source = estimates[0][0], estimates[0][1] + " (first variate; set SEASON_PERIOD to override)"\n    last = P.last_value_baseline(context, horizon)\n    fit = least_squares_forecast(context, horizon, period)\n    baselines = {\n        "last_value_baseline": {"mae": P.mae(truth, last), "rmse": P.rmse(truth, last), "what": "repeat the last observed value"},\n        "least_squares_trend_season": {"mae": P.mae(truth, fit), "rmse": P.rmse(truth, fit), "what": "level + linear trend + one sine/cosine pair fitted to the context by least squares (4 parameters per variate)", "period_steps": period, "period_source": period_source},\n    }\n    reference = {}\n    if data["sample_kind"] == "synthetic":\n        _series, signal = synthetic_series()\n        noise = truth - signal[None, -horizon:]\n        reference["noise_floor"] = {"mae": float(np.mean(np.abs(noise))), "rmse": float(np.sqrt(np.mean(noise**2))), "what": f"the generator\'s own N(0, {NOISE_SD}) noise on the holdout: no forecaster can expect to beat it"}\n    np.save(run.state / "least_squares.npy", fit)\n    run.write_state("validated.json", {"baselines": baselines, "reference": reference})\n    for name, row in {**baselines, **reference}.items():\n        print({name: rounded({k: v for k, v in row.items() if k in ("mae", "rmse", "period_steps")})})\n\n\n# --------------------------------------------------------------------------------------------------\n# model stages (CPU reference path)\n# --------------------------------------------------------------------------------------------------\n\n\ndef cpu_pipeline(run: Run, P):\n    os.environ["CUDA_VISIBLE_DEVICES"] = ""  # before torch is imported: the reference path is the CPU\n    return P.TiRexForecastPipeline.from_pretrained(device="cpu", weights_dir=snapshot_dir(run, P), allow_download=False)\n\n\ndef stage_forecast(run: Run) -> None:\n    import numpy as np\n\n    P = package(run.root)\n    data, series = load_series(run, "forecast")\n    horizon = data["horizon"]\n    context, truth = series[:, :-horizon], series[:, -horizon:]\n    pipe = cpu_pipeline(run, P)\n    result = pipe.forecast(context, horizon=horizon)\n    np.save(run.state / "quantiles.npy", result["quantiles"])\n    summary = {k: result[k] for k in ("point_forecast", "horizon", "context_length", "n_variates", "device", "source")} | {"quantile_levels": list(result["quantile_levels"])}\n    run.write_state("forecast.json", summary)\n    print(summary)\n    q = result["quantiles"]\n    for step in range(min(5, horizon)):\n        print(f"step {step + 1:>2}  median {q[0, 4, step]:.4f}  truth {truth[0, step]:.4f}  q10 {q[0, 0, step]:.4f}  q90 {q[0, 8, step]:.4f}")\n\n\ndef result_from_state(run: Run, needed_by: str) -> dict[str, Any]:\n    import numpy as np\n\n    summary = run.read_state("forecast.json", needed_by)\n    quantiles = np.load(run.state / "quantiles.npy")\n    return {**summary, "quantiles": quantiles, "median": quantiles[:, summary["quantile_levels"].index(0.5), :]}\n\n\ndef stage_evaluate(run: Run) -> None:\n    P = package(run.root)\n    data, series = load_series(run, "evaluate")\n    validated = run.read_state("validated.json", "evaluate")\n    horizon = data["horizon"]\n    context, truth = series[:, :-horizon], series[:, -horizon:]\n    result = result_from_state(run, "evaluate")\n    report = P.evaluation_report(result, truth, context=context, sample_kind=data["sample_kind"])\n    ls = validated["baselines"]["least_squares_trend_season"]\n    report["baselines"].append({"id": "least_squares_trend_season", "metrics": [{"id": "mae", "value": ls["mae"]}, {"id": "rmse", "value": ls["rmse"]}], "what": ls["what"], "period_steps": ls["period_steps"], "period_source": ls["period_source"]})\n    if validated["reference"].get("noise_floor"):\n        nf = validated["reference"]["noise_floor"]\n        report["reference_points"] = [{"id": "noise_floor", "metrics": [{"id": "mae", "value": nf["mae"]}, {"id": "rmse", "value": nf["rmse"]}], "what": nf["what"]}]\n    n_points = truth.size\n    report["coverage_granularity"] = f"q10–q90 coverage over {n_points} holdout point(s) moves in steps of 1/{n_points} = {1 / n_points:.4f}"\n    model_mae = next(m["value"] for m in report["metrics"] if m["id"] == "mae")\n    last_mae = validated["baselines"]["last_value_baseline"]["mae"]\n    lines = [f"TiRex median MAE {model_mae:.4f} vs last-value {last_mae:.4f} and least-squares trend + season {ls[\'mae\']:.4f}"]\n    if validated["reference"].get("noise_floor"):\n        lines.append(f"noise floor {validated[\'reference\'][\'noise_floor\'][\'mae\']:.4f}: on this synthetic series a 4-parameter fit of the right family already sits near the floor, so beating last-value says little about model skill")\n    elif model_mae <= ls["mae"]:\n        lines.append("TiRex beats the trend + season fit on this holdout; repeat over several holdouts before reading it as skill")\n    else:\n        lines.append("the trend + season fit does at least as well as TiRex on this holdout; a simple model may suffice for this series")\n    report["interpretation"] = "; ".join(lines)\n    run.write_output(f"{STEM}_evaluation_report.json", report)\n    run.write_state("evaluate.json", {"model_mae": model_mae, "interpretation": report["interpretation"]})\n    print(json.dumps(rounded(report), indent=2))\n\n\ndef stage_export(run: Run) -> None:\n    import importlib.metadata\n    import platform\n\n    import numpy as np\n    import pandas as pd\n\n    P = package(run.root)\n    data, series = load_series(run, "export")\n    horizon = data["horizon"]\n    context, truth = series[:, :-horizon], series[:, -horizon:]\n    result = result_from_state(run, "export")\n    baseline = P.last_value_baseline(context, horizon)\n    fit = np.load(run.state / "least_squares.npy")\n    rows = []\n    for v in range(result["median"].shape[0]):\n        for step in range(horizon):\n            row = {"variate": data["names"][v], "step": step + 1, "median": float(result["median"][v, step]), "truth": float(truth[v, step]), "last_value_baseline": float(baseline[v, step]), "least_squares_trend_season": float(fit[v, step])}\n            for index, level in enumerate(result["quantile_levels"]):\n                row[f"q{int(round(level * 100)):02d}"] = float(result["quantiles"][v, index, step])\n            rows.append(row)\n    pd.DataFrame(rows).to_csv(run.out / f"{STEM}_forecast.csv", index=False)\n    source = json.loads((run.root / "source.json").read_text(encoding="utf-8")) if (run.root / "source.json").is_file() else {}\n    payload = {\n        "forecast": {k: result[k] for k in ("point_forecast", "quantile_levels", "horizon", "context_length", "n_variates")} | {"median": result["median"].tolist()},\n        "evaluation_report": json.loads((run.out / f"{STEM}_evaluation_report.json").read_text(encoding="utf-8")),\n        "input_manifest": json.loads((run.out / f"{STEM}_input_manifest.json").read_text(encoding="utf-8")),\n        "sample": {k: data[k] for k in ("sample_kind", "name", "shape", "float32_sha256")} | {"trimmed": data.get("trimmed")},\n        "notebook_source": source,\n        "repository_revision": source.get("revision"),\n        "model_id": P.MODEL_ID,\n        "model_revision": P.MODEL_REVISION,\n        "model_license": P.MODEL_LICENSE,\n        "runtime": {"python": platform.python_version(), **{d: importlib.metadata.version(d) for d in ("torch", "tirex-2", "numpy", "pandas")}, "device": result["device"]},\n    }\n    run.write_output(f"{STEM}_result.json", payload)\n    print({"rows": len(rows), "columns": list(rows[0]), "runtime": payload["runtime"]})\n    print(sorted(p.name for p in run.out.iterdir()))\n\n\ndef stage_activity(run: Run) -> None:\n    """TRX-M2: forecast from a shorter context and print MAE and the mean q10–q90 band width beside the full context; writes\n    only to outputs/activity/."""\n    import numpy as np\n\n    P = package(run.root)\n    data, series = load_series(run, "activity")\n    horizon = data["horizon"]\n    context, truth = series[:, :-horizon], series[:, -horizon:]\n    full = result_from_state(run, "activity")\n    length = int(run.options.get("context_length", 64))\n    if not P.MIN_CONTEXT <= length <= context.shape[1]:\n        raise ValueError(f"ACTIVITY_CONTEXT must be between MIN_CONTEXT={P.MIN_CONTEXT} and the full context {context.shape[1]}; got {length}")\n    before = sorted(p.name for p in run.out.glob(f"{STEM}_*"))\n    pipe = cpu_pipeline(run, P)\n    short = pipe.forecast(context[:, -length:], horizon=horizon)\n\n    def summary(result, n):\n        q = np.asarray(result["quantiles"])\n        return {"context_length": n, "mae": round(P.mae(truth, result["median"]), 4), "mean_q10_q90_width": round(float(np.mean(q[:, 8, :] - q[:, 0, :])), 4), "q10_q90_coverage": round(P.interval_coverage(truth, q[:, 0, :], q[:, 8, :]), 4)}\n\n    rows = [summary(full, context.shape[1]), summary(short, length)]\n    for row in rows:\n        print(row)\n    run.write_output(f"activity/{STEM}_activity_context_{length}.json", {"rows": rows})\n    if sorted(p.name for p in run.out.glob(f"{STEM}_*")) != before:\n        raise RuntimeError("the activity changed the canonical outputs; it must write only to outputs/activity/")\n    print(f"wrote outputs/activity/{STEM}_activity_context_{length}.json (the canonical outputs are unchanged)")\n\n\nSTAGES = {\n    "weights": stage_weights,\n    "data": stage_data,\n    "validate": stage_validate,\n    "forecast": stage_forecast,\n    "evaluate": stage_evaluate,\n    "export": stage_export,\n    "activity": stage_activity,\n}\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--outputs", type=Path, required=True)\n    parser.add_argument("--weights", type=Path, required=True)\n    parser.add_argument("--stage", choices=sorted(STAGES), required=True)\n    parser.add_argument("--options", default="{}")\n    args = parser.parse_args(argv)\n    run = Run(args.root.resolve(), args.outputs.resolve(), args.weights.resolve(), json.loads(args.options))\n    error_file = run.state / f"{args.stage}.error.json"\n    error_file.unlink(missing_ok=True)\n    try:\n        STAGES[args.stage](run)\n    except BaseException as exc:  # noqa: BLE001 -- every failure is reported to the kernel with its own message\n        traceback.print_exc()\n        error_file.write_text(json.dumps({"type": type(exc).__name__, "message": str(exc)}), encoding="utf-8")\n        return 1\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'requirements.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt\ncertifi==2026.7.22 \\\n    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n    # via requests\ncharset-normalizer==3.5.2 \\\n    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \\\n    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \\\n    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \\\n    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \\\n    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \\\n    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \\\n    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \\\n    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \\\n    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \\\n    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \\\n    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \\\n    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \\\n    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \\\n    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \\\n    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \\\n    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \\\n    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \\\n    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \\\n    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \\\n    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \\\n    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \\\n    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \\\n    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \\\n    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \\\n    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \\\n    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \\\n    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \\\n    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \\\n    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \\\n    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \\\n    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \\\n    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \\\n    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \\\n    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \\\n    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \\\n    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \\\n    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \\\n    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \\\n    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \\\n    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \\\n    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \\\n    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \\\n    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \\\n    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \\\n    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \\\n    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \\\n    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \\\n    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \\\n    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \\\n    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \\\n    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \\\n    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \\\n    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \\\n    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \\\n    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \\\n    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \\\n    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \\\n    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \\\n    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \\\n    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \\\n    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \\\n    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \\\n    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \\\n    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \\\n    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \\\n    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \\\n    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \\\n    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \\\n    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \\\n    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \\\n    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \\\n    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \\\n    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \\\n    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \\\n    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \\\n    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \\\n    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \\\n    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \\\n    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \\\n    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \\\n    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \\\n    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \\\n    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \\\n    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \\\n    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \\\n    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \\\n    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \\\n    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \\\n    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \\\n    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \\\n    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \\\n    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \\\n    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \\\n    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \\\n    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \\\n    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \\\n    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \\\n    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \\\n    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \\\n    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \\\n    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \\\n    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \\\n    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \\\n    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \\\n    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \\\n    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \\\n    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \\\n    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \\\n    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \\\n    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \\\n    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \\\n    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \\\n    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \\\n    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \\\n    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \\\n    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \\\n    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \\\n    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \\\n    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \\\n    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \\\n    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \\\n    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \\\n    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \\\n    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \\\n    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \\\n    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \\\n    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \\\n    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \\\n    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \\\n    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \\\n    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \\\n    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \\\n    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \\\n    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \\\n    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \\\n    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \\\n    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \\\n    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \\\n    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \\\n    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \\\n    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \\\n    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \\\n    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \\\n    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \\\n    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \\\n    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \\\n    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \\\n    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \\\n    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \\\n    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \\\n    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \\\n    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \\\n    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \\\n    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \\\n    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \\\n    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \\\n    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \\\n    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \\\n    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \\\n    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \\\n    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \\\n    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \\\n    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \\\n    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \\\n    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \\\n    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \\\n    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \\\n    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \\\n    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \\\n    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \\\n    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \\\n    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f\n    # via requests\ncontourpy==1.4.0 \\\n    --hash=sha256:018227e134b73090b06f911e773a526c77c9af426be29065500ce096a5accd4c \\\n    --hash=sha256:072a702e2e178f4fcf96f04775d0c059e4c917925abf0d3208ebb6865df3c23d \\\n    --hash=sha256:0c7a4c2716a4e98342221954416a836cca77c996c14ddbf22b5f01d5d93ca09c \\\n    --hash=sha256:107ec46f7aa1664266d69b1181aadc953d58d39f7d8d648fa5b795d4a060b0de \\\n    --hash=sha256:181bea01bc742734ae672fa00c717d855dd35e1f029536406538c52e7b0cd73d \\\n    --hash=sha256:186ba929df36d61b6127da2e89cd1357e4cb79aca647381ab1a6feb0b152877b \\\n    --hash=sha256:196759a3e4db60e1546167e361090ebb6d199aa3aafc9e34ea17a5ac3b23e814 \\\n    --hash=sha256:20156f5a1ac4f8ce02656e39a61e82164a3d359796dc8026f75b062783d500e1 \\\n    --hash=sha256:20da2f86bfaed60dbba729fe738c8241b1dd22b6cf7e8cbbf30df290bd04ba28 \\\n    --hash=sha256:2deb580178ca19437a84bd77e4bc2cd91a8ccad212413a83c273900868b5978c \\\n    --hash=sha256:2e741a39dfc96babe1722561e81351aeec104526c57dbe91c167c1db606c2d55 \\\n    --hash=sha256:3863ef2e2b13fe93f8c0ebb08ee400cb07153b8b0e91c5acb26e4537f283634f \\\n    --hash=sha256:3e0a2392533e5e2abdf277c951047516acc8d2f3c42b2dd9dc34c907c9ebabf2 \\\n    --hash=sha256:404dbcd9233513dfd1323b66ea593fef90e899f658f6b8a9ed8e93bd0ca669db \\\n    --hash=sha256:417be048f7e122cefbe2a34c51a1f2b0410f8eb35796f794d45475ddb7872d9d \\\n    --hash=sha256:423bd5f4b3f11d54a8c597234e513382e3454bb106b8ec7ae32ba4ed63f8d99b \\\n    --hash=sha256:432e89f835cf01f8d2123a130a3126b17ee6e0348e4d62c0973872b30aa873bf \\\n    --hash=sha256:43c3ccbb32c6294b183dcc8e8c46dacf5ecef809497e3d48a5be298eef185ad0 \\\n    --hash=sha256:43d80072a32299bf945de0a6dd4e034ea0162e832261de2e07a274a5adbba9c9 \\\n    --hash=sha256:4bffcb2e8cc5c0e837631acab59dd04265fdac376550e7c3e310523877c68cb7 \\\n    --hash=sha256:510f7d93d94cf6ebf4e2cd0640bea16ab8affac7547230175b6745b596ce0599 \\\n    --hash=sha256:5450f091ac1be0be3ad3a2a3b3f23b5e443e78c670ced4fd347d626f92a28fd2 \\\n    --hash=sha256:5545ff38229c15d11d822fc6f7415b932da9f4975f53e305b7b96ebfb01f889c \\\n    --hash=sha256:5a4c89c7f38a0d7a94356d74e3391073c4325c4d47ad2fd065c3fe7dbae16c0c \\\n    --hash=sha256:61624f6722480aa7e168fd746164e7bfdf48f8ccb6542f1613741200dc37e2b1 \\\n    --hash=sha256:618137ec5778fb76d494c9ec854b104a377fddba0d128bef7136cce2c3bf0df8 \\\n    --hash=sha256:630d32f06cedf37b7f1dd2b6a12e4492826bd477a4c9c7b2e2fee6ce3a2cea76 \\\n    --hash=sha256:64039341e2d8804f1a13bda8c69083eab935167bbd7e856cf096241f17d5bd45 \\\n    --hash=sha256:6507a016976a75a15aee47809a34605de6244ccc056e6164bf9fb14b27eecad4 \\\n    --hash=sha256:6ced1670fafee703b8277ab1645072a226e74f167166eae146fb743916119b67 \\\n    --hash=sha256:6e697d94e69f499ff6bebb899cae97a58d5d14f0e1fe9568b43a0248d2f9af8c \\\n    --hash=sha256:6f963ea9f0d68d2b6a02d861e3c0aae09ae25ba21f243fcf281e9fed468b078d \\\n    --hash=sha256:738c44fa71735a617f36da58e32810512407d283d5d1bb5b1cecb00d7eeba7cf \\\n    --hash=sha256:758e7496cec3195fc28a28945eab3ceef348c47fb95c7afc2eca2f3901209c8c \\\n    --hash=sha256:78ed5c5f962b3e156109f0531b174a65ba80d13cc681a70227b12524c162e184 \\\n    --hash=sha256:79b06c60d5e569ce12c5da8f0c51217cdc484386e2349fa717c30fafb56d2871 \\\n    --hash=sha256:7f861af508fca2384eef392bc1d8377cfa349b2b854cb11b45d89c75780e2561 \\\n    --hash=sha256:80c7fc8e7ac217777ce7ba2db3cf8478652b5a4b335283111012ff9eaac9d842 \\\n    --hash=sha256:86d05cec773c9507a3950122e0e40ce77c23c75ceeb2fc189514e71893cbb34b \\\n    --hash=sha256:875f42444c9cf48d56f724f2637e60d0f73b3b12c9041e1484580a233edf9591 \\\n    --hash=sha256:90feb8da006803a95ba573fddc9536357d0c3faddc3a4c9e22816ca49af52878 \\\n    --hash=sha256:912c6afaa106e2f74ba22b30416b77ef7eb94e3bdc5a4df51ab147845dba9b6d \\\n    --hash=sha256:96019f059bcb3774acc0104be3360a57b36d33eaaa2e1d7f77c800ca8e07618b \\\n    --hash=sha256:9c0e07c691f3b3321913ed9b8161c50ea5f77fadd006f52f5e755359b0dcbfc5 \\\n    --hash=sha256:a03b32c2e7eda8b17757c022162c13c86f5c3d6f937dddf9ba3c3ff7b513953b \\\n    --hash=sha256:a1c8a74744fa746eeaa12f0f9ed7f8b4add9d8f1b14d95e3b5e133675eac888f \\\n    --hash=sha256:a3e67bc1a6a4618a1dac7c3053a9ffece5ddfa2046b2670b342cf430bb0b87d1 \\\n    --hash=sha256:a58b5f130afec61a093d743cdda435cc4047748b6791627e34a68f41ec3a9b07 \\\n    --hash=sha256:a5e2bb871b90cd7a52bdee63bf80ef7acb46398d27efc66d7679240016efc5aa \\\n    --hash=sha256:abd0f7e51ecc52bf8c95713bd80b1c17c516a71fa391c54475308a4377903c2b \\\n    --hash=sha256:b7794ea07cab575633daad8d6e963b662053391022294d3aacc1d9ba9ef54114 \\\n    --hash=sha256:b9dc493e7924e5aa32d89d0b8e50986280f3d4b284968a792787bd7bcffd3cad \\\n    --hash=sha256:be85d160e7c795113c2a93254dde2fc7e86b375322570eee78b420969426973c \\\n    --hash=sha256:bee6ff96653e0807cd0f8eaef00174a031e3a2effa5c20f49a5a8d574b05af23 \\\n    --hash=sha256:c76f3a5318164db7d9401132fc1b5364b784c613c93fa506e3b5e6d1bf353ec8 \\\n    --hash=sha256:c8be3392b84cf43373a488874dc1ebf46f60784a70977f92586e18019d71c7ed \\\n    --hash=sha256:c927ec747633e68c9920bcaad48b0bcefe648812b822f18c34e2d0e2270a3d2a \\\n    --hash=sha256:cc87f2ffa84a49a77d76cb792f28a48ff3b05a222aa0b2bddbd839ba78e1d5ca \\\n    --hash=sha256:dd59df9e2fb0aff8bd7fd1adb9f349b7c835245a9d0f18c2f2deeb537190f2b1 \\\n    --hash=sha256:ddf5a2596d716fd3793434844caf31abc7a40b1e8718431420c89858268fb909 \\\n    --hash=sha256:de503609fdb71f597634ca64fb4bd2f13d27c97f140e912b1e8ed93544840df0 \\\n    --hash=sha256:e21d1d4a9db5b3a793649c7ab0ddb4697f2818929eec871a7b9c45fce7b2a1ea \\\n    --hash=sha256:e439ab450c93455feb0218532ded3e946ec7a9b5f459069f122cfa44b89229f6 \\\n    --hash=sha256:ea09dc704029930cbd097f75cb0cc1db9852adbcf1a151fac7e9559f7bcbfd19 \\\n    --hash=sha256:ef47fef9a912c77e3d84b014f701e5a9340f25703e9ed3bcebe37f91fb69dc49 \\\n    --hash=sha256:ef9440f6f8506246269a82734f5ff9e2e4c5e775b3996fc883cc491c5257eca6 \\\n    --hash=sha256:f1219a8898523cba821085f2da8a1b962cc696325763f09d7f93538ba43b2d70 \\\n    --hash=sha256:f863c6100bf926cf47d13f3cd75f9bb8ebd98aaab230eeb4468b91f98f39a6b3 \\\n    --hash=sha256:fa1b787362a3856e63dd2b89449f6c384d55ee1beb8f94f4bcc871b40f02462c \\\n    --hash=sha256:fc9feef8f1f001c5b87decadc67c4a5d1eebb62ca39c4763d1237ff62cf2b707\n    # via matplotlib\ncycler==0.12.1 \\\n    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \\\n    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c\n    # via matplotlib\neinops==0.8.2 \\\n    --hash=sha256:54058201ac7087911181bfec4af6091bb59380360f069276601256a76af08193 \\\n    --hash=sha256:609da665570e5e265e27283aab09e7f279ade90c4f01bcfca111f3d3e13f2827\n    # via\n    #   flashrnn\n    #   mlstm-kernels\n    #   tirex-2\nfilelock==4.0.12 \\\n    --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8 \\\n    --hash=sha256:cf42711a7ac791818b299fab0332a088c65aeeefa36290de98db92c434303b0c\n    # via\n    #   huggingface-hub\n    #   torch\nflashrnn==1.0.8 \\\n    --hash=sha256:3615b2f9dafa59d85f40cc6afbbbab32814eb09b581f6931d91f1777e747f95d \\\n    --hash=sha256:5d36168fac545af8513ffcb468f1b296b7b7ccb40f8153a3a062d170fee2c346\n    # via tirex-2\nfonttools==4.66.1 \\\n    --hash=sha256:058cd823b80bac59e64dfad9e3b6fcd677852f9a3804971bbf6b48cc611e785c \\\n    --hash=sha256:05aeb146451f37289f782c3c861f3d0f4b86c2dd2e4620b46683544c7406640e \\\n    --hash=sha256:05c0fff6b4a5d872ed89cab2c4f81060b86ace263903eb4e8d0edcac47a60dfa \\\n    --hash=sha256:08d8956e3ec990c75230d92f1630b215e8f3738c83a003421c22b31ebfd0ce15 \\\n    --hash=sha256:09ae73bd219e1245debd8376077a0fa6e03175e255c4f51bae5f6a271bfe384a \\\n    --hash=sha256:0dc6fd99cb8c30941036308b148da9432640442a6f26f36d71dad9be24cbd0e9 \\\n    --hash=sha256:1801fdad5600118327171e0e8aa79f7cc48831dd55ab36998c9de03bd5ffe6cd \\\n    --hash=sha256:261d8dc95845e751f975fe8d6075600593ee253470d46d1b84801688051b09f6 \\\n    --hash=sha256:2aeb745f2664eb811026997c95628071137a777ea2ad296deec9cb393f0b23cf \\\n    --hash=sha256:2c7340497cf53490293e0c2b61011e0191633022ede0a0a964a68157a98b0fb4 \\\n    --hash=sha256:2ce4c93160535761f22c80b2afbc96cabc09855363a5d1a5554265b8a4c85901 \\\n    --hash=sha256:2d320483928c7831f0139ecb361954a26b2e2a8995681200155835dd8cd4a7d5 \\\n    --hash=sha256:2d637468dac23aac0e223bd52e66f8faa3b0dfcef57435460fa2107e830226cd \\\n    --hash=sha256:3087a430722aba8de429c2539fd2a58a9cf05238cdfefd8626460001052ca878 \\\n    --hash=sha256:34378db9a398b59de18cc79d942f0a907c6fc6301945e065ec888202f607aa3f \\\n    --hash=sha256:36bb24d4b98faacaff04af1d5e0a4285feba6ed1da6728cd34b6b6deb6bbb934 \\\n    --hash=sha256:38ce8f5fbd5c17dd2153d47d7c8d4108f3deda3f2b4a79b60ddc470a58faded3 \\\n    --hash=sha256:43d1284c1964666ee833f2badd3017dc138f53d4889043ffca66c5ce4188f188 \\\n    --hash=sha256:53e5854ea8003efec34adc0863c18ce91da923018354d27366f7fee7db928d7a \\\n    --hash=sha256:56d41d650cb8fc6cfe1d85ed7c62a0a56cbeed07bc65ca795475b914d401312a \\\n    --hash=sha256:5de5d80fbc0e50ff794c244e8fb7afd3eadfe0fa232ba8b162b8c551df22fcb4 \\\n    --hash=sha256:60f5ea17aed4262630afa43f26997ceabd6417fa05dcedf54c665f5a29193e18 \\\n    --hash=sha256:64967c6ddb0d4c610dfd8cb1485981b2d27972ddfb7d4bbbd9e199d2a089c450 \\\n    --hash=sha256:668f092bc0de8902167df6a0d5c5aedc3b4f9e43cf88eea92e9b46a2bd3968f5 \\\n    --hash=sha256:66fad3b7874062c2a2692f0ae6dea56d24f01b778c7f191950ca3ff997e25a88 \\\n    --hash=sha256:6946fe7bfb28590a1fd4061a17609c9a843952deb65dcf30d1fe725070c3e7a4 \\\n    --hash=sha256:71c7ca1b5f46f5dd549f56b47d47c0b709217675c23d3a7bc6aa1a69b6d9bbae \\\n    --hash=sha256:72299346b96b9244dabcc051b24e4653da4edfda6105544cfb10ce856a1afaac \\\n    --hash=sha256:7234ae9e28db64273fbbfa72caebd0a97e3bdba6b05064114741b9539ef339d0 \\\n    --hash=sha256:7b8ff9e0edbcee2fbf7dff0c41b9041c1901c26acf64e23adb67495012df11de \\\n    --hash=sha256:7cf4f996f9b1cb549bff9ea4c50813988a26ec922c95cfa85c7e4f1270447e06 \\\n    --hash=sha256:7f49f2834f5d006fe0f3bb10fec73b261806c50941f0cfbc08294074ffc32210 \\\n    --hash=sha256:83572afe48733bad7a4a9c11721d3a726c2e976d82b063fc9bdd049d76955abd \\\n    --hash=sha256:8526b2b7ec4db6b81efb83438be52b1264eda9a4994d867163cfe8c65581ce8d \\\n    --hash=sha256:8aed2bbcd6216253ef1b015763593365ee8084f621dfa53bb957c19d5e05f7cd \\\n    --hash=sha256:90de3477394c73481d27d2b86091c1c736053ee13ff52c42f0e151948e8578c6 \\\n    --hash=sha256:9261ef507f2dd74203443a472b65b5a26429eb378f975016dec7dc7305b24898 \\\n    --hash=sha256:9ea6c93091cbf83161a544388746a0911550bd98cb911faca3591cf5ead166ac \\\n    --hash=sha256:b13c8c541ce0b794add3211b3641cc0e113d707f73e06235e6fe9731bd7c45a9 \\\n    --hash=sha256:b18803cbdef248e7ee1be59cb277fbbe1da1faaa6f726fa5d3557904e6a3d967 \\\n    --hash=sha256:b878c78b2af11b879bd4f26bb0d8bda2a4c64543fdd3f28efe2c80f97f043885 \\\n    --hash=sha256:b8b71db96d605784e2c5ebf0788a406018ea8fdd80338491f4c83613d5cd1fec \\\n    --hash=sha256:b913b8e9f7ca9bec44d1eb919f591c596c61041aa357c96be55ff93169859e91 \\\n    --hash=sha256:c258eba62260beb33c110b03a6912cefa3635239c4ab5615b7225fb6f7b85238 \\\n    --hash=sha256:c47299bca4b5acaaeb32100f77b944feea151de9ef1773365a410dc3d49b945b \\\n    --hash=sha256:c666fefdd5613a0e99aa4516e6ff4ef87aa86cf1c7ba12a73550f4770e46b750 \\\n    --hash=sha256:c724e56213494c6695335577822b2d1628d102e71614de8b7eb8e30886d6a314 \\\n    --hash=sha256:d3b5403e82d0c7659ff1d9f956e29a3a68d094f043e9f5bc0442796fc3a4fb58 \\\n    --hash=sha256:d4f76868aea9cc4ce47fdbeaa904c02ee7d85dd0ad095071ae77f0bda6e62cf5 \\\n    --hash=sha256:d84ac0bf776b68396185bd919dd29e633d94300660335efc40b55b294b886903 \\\n    --hash=sha256:d8f0a8f16c4f3a5a87ca971de2631792d8cb4d570951f2000acf712f157d40db \\\n    --hash=sha256:dbb7b950f8c02deaffb6968994691e8589d671b7ef8396bc9d5b5c0dfbb7292f \\\n    --hash=sha256:dfba62cc93199ba62c376f90f2a9147d92730d301e44f88e013e50ff5edf6193 \\\n    --hash=sha256:e1cde50b3ec84ca6fe63ca815de183dbecb88e8adf8ada82d8ea130ef12b2b43 \\\n    --hash=sha256:e7ea7a08547a453fa000db96ed5714a3dc7e2b4255b9243f897921f8c10c169a \\\n    --hash=sha256:eef76d5796e604f9d6753fa6d323c4eb9f4e0e43f1dcca553f3e6914f1667b64 \\\n    --hash=sha256:f08ab7f8461c37ecfdd29ad97fb0c0780b50501bd664bb0f46b6e83ed2b9d2a7 \\\n    --hash=sha256:fdf4afd75c643e60ef4a96fe64fc8a9def27d2a542112332371a9e5066885f9a\n    # via matplotlib\nfsspec==2026.9.0 \\\n    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n    # via\n    #   huggingface-hub\n    #   torch\nhf-xet==1.6.0 \\\n    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n    # via huggingface-hub\nhuggingface-hub==0.36.2 \\\n    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n    # via\n    #   tirex-forecasting-pipeline (pyproject.toml)\n    #   tirex-2\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via requests\njinja2==3.1.6 \\\n    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n    # via torch\nkiwisolver==1.5.1 \\\n    --hash=sha256:007a5553dfc4f4e8d184f588a0200e2cd4b63a59cc8796df3c39909e679dc7a0 \\\n    --hash=sha256:0324cd2567259b7a095f6cf18a52b0ffc6f3de9e69528ff1bc0e7a37bd43ff1a \\\n    --hash=sha256:0627b9bceb9c3cdcf12b8a18655eedfed2692b038df27423383c120d0b7dc2d6 \\\n    --hash=sha256:06a6917674de9e0fe3f66f5430787f59a9f2ddb64af9b714eaec547e29ef5c19 \\\n    --hash=sha256:072bdb15a3c19a5b5dbc8f8fb1f4e1884bf4f3507eeb4cc6334401274d37a5c0 \\\n    --hash=sha256:0a4faea5c6db201c6a21391d2ac926ea97acf7dacdbc3c417189e1adb1a00837 \\\n    --hash=sha256:0ba9527afc80ae3d7814ed98b6572d02bf85eaf48065678342c5f0c6dab7a8c7 \\\n    --hash=sha256:0d8924877ce22e17326a99a418c3c82037da078df3c6a260b13eca677444e6e7 \\\n    --hash=sha256:0ebdef3eae5336568147c39a55be6a2036ffde53faa9ca2d978989ae7c2da12c \\\n    --hash=sha256:1209042a623ddfda5497e4066c7b77651dde8e1d3a9dd97599dc7e97f3b9b78c \\\n    --hash=sha256:16895f553ee6620a827d2da56b871f835fb70b9216cca5d188e885caf6e3bd23 \\\n    --hash=sha256:17851e5dad4484be0cbccbde3b15331deae036de9aebd45eed964487802b172f \\\n    --hash=sha256:1798e83840c3f627246104c4d8a9639c60fa068adf9ce92b61791781fa8a68c1 \\\n    --hash=sha256:18170a77ddfecf40ec60d0928268dc95880c881864e015a8f34094ed18b9b9ad \\\n    --hash=sha256:186884a58486651e3c217b6acea0a53eaa9498fdd472057c46f2f0fb5c25aad5 \\\n    --hash=sha256:18a0cfb124546a4c2e6087c5f3029c7f44b37c85b142e0ced71f73a7599ac208 \\\n    --hash=sha256:1983f0974a750a6f6556f368ba11105d1d8369c735b944747c9f12ae5aea7aae \\\n    --hash=sha256:1a7587dc335f2c0f5bd577fd0540bd16c66006bdb60f759a1059f025e6c4f071 \\\n    --hash=sha256:1acc7e5b7ef05e9da8bb70cd6c7c4513090213d2e1ad9720f599f0bf6c52aec5 \\\n    --hash=sha256:1d852545c4d0e35a72728d072cbaa59e2fa7dd84bdf01e068d670dd0ceb58eb6 \\\n    --hash=sha256:1ed0f5e49d0ceff8b72190824d9e59c062fbbc02c231b853112c78474b3f5ec2 \\\n    --hash=sha256:1fff05e239575b1481b6ed1a782f6fad616efbf1f0b1f44e6e85c4dfe426e483 \\\n    --hash=sha256:21e46b23a2da695c364124817bc01d970effd5483147f8d66a6a7167e3f6b851 \\\n    --hash=sha256:22d5e5aaad6be121f2515765e3b1c444352cb8eb4c86510801db8f2e50757316 \\\n    --hash=sha256:2551cf9917af48ee7c4b29cc82320489508cf96fd26a51f6fc124de661cd44c7 \\\n    --hash=sha256:255605693a483db7bd5c79f60437f7bf658f7f520d61aa42722e32257c941951 \\\n    --hash=sha256:26e8268480be5061d509e29669d59103c067a26377a56491630ece11762e3858 \\\n    --hash=sha256:27add358abe374ebaa3b8763ef380bc99051b5a4b18d94878366a9e4f59efef0 \\\n    --hash=sha256:2ae70bc59790d2af72a3f76f24b272403e135070340281108b447cb77ea70819 \\\n    --hash=sha256:2e10ae1bba1899188b33557c10d73affcc12033edd18adddb57d209039976a4c \\\n    --hash=sha256:3221f78211074f561c44ca42eac0619828171bec15a2c4cf6f7747d07df76e8e \\\n    --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38 \\\n    --hash=sha256:34ec467940442c9943016fb2d4c81d1ba84351eeca2f1a78f8bc87f1ba0d414c \\\n    --hash=sha256:37f801b5d7cc0e5a548921308e059fd2b057bb42972b591cfa3049f95423c4ed \\\n    --hash=sha256:38f6e0deb4d0a4615efe0c4efc5990b06ae450ab50a0b321c0b078b6d238c083 \\\n    --hash=sha256:3c24cd69455e1b00ddf770c13b6e2c33e07d6dc3f2d34add0bf9277c5c6bbd46 \\\n    --hash=sha256:3cc210010fd2f438a3ed430b45f1b501fd13a8618bf984dc2c5ce5b69b78752e \\\n    --hash=sha256:3fa5855898f6d3d01b72ccd48a2d65cbdee301251603fefe34e2025bddba219c \\\n    --hash=sha256:416ba7ff9f233b7036689bb5a3783537e838ad483f63558d2a800f75afe738b1 \\\n    --hash=sha256:431dc224a1a92a5c8f582d96e505196a3b5997a7271076678da2dfde67b77e9a \\\n    --hash=sha256:43844c1a7ad6d723d5b5b4c4fc7f5bd399c40e288120d16257c7c9e8765c6e85 \\\n    --hash=sha256:44b8faef94f1857e77fa0238f3390ff1ac51d2ea20a487e2e452a59fd2b5f5ca \\\n    --hash=sha256:470d420f98d368d6f010633a20659b544c5fdfa5329e6b70219f2ef08fd4a7ef \\\n    --hash=sha256:482676e5bd48d70ac99d9fc78863469845421e01184fa83f1f9366dc49f7e974 \\\n    --hash=sha256:4d4ca09bf13cff792b1884f64b98ee6c2467930d632233be25c56b442d99f10e \\\n    --hash=sha256:5025e36fb4fb275cef0a4e30dbb11cb4ae61d1c83deb90189cb5d7e4cafd6b55 \\\n    --hash=sha256:509735237ae0d849e8a843551d423d2500d2e0a9ac1611a145658b29c0fb9f85 \\\n    --hash=sha256:534f02c1abb31ed6dbd3515545285c330b2f12d00fdb1fdb71658b9ca5a13a6a \\\n    --hash=sha256:5978c3340f16a35c30f8ab2fa7bcf559973c55f1a5ef6970e1f621acf3c4db13 \\\n    --hash=sha256:5b973887ff782cfd6b67c9904ad8ca542e0bc5e4961503408b423b5a688b4d38 \\\n    --hash=sha256:5c490db2168a508088f59140dd392556a54b8bd1048fc6383c8baff13c359673 \\\n    --hash=sha256:5d142e352eb13facc7dd047489aebdff6ba78576c239f1ea04931979caaf0567 \\\n    --hash=sha256:5daa1f19e097050b9c4d9a78fcc9263cb96c9dfae08037ddc1b7c4ad1889f2a2 \\\n    --hash=sha256:61e9a64c7635095a6bfe483e2ff055d437c59bd45f3617a228b37277f0185d62 \\\n    --hash=sha256:63fb7294b768f444eb4b068965f2662f28c2fd4161e23bd60fcf3ff27b74c046 \\\n    --hash=sha256:685929988b208a911f1285e2f8ed54210b0d681a3dc0f03e00d599d291986e7e \\\n    --hash=sha256:6a797a1cefc8b9c93170db580337e1fe3d011ad18b1299943231279406342048 \\\n    --hash=sha256:6b92f60017dda7d877fdc546438b5e28f31c523264f49cf5a48c1d0ce1a0dfbc \\\n    --hash=sha256:70ed9a45c7484d2b30cdacf60d220f494a1763b9fec1ad03285c6553fa0889f2 \\\n    --hash=sha256:719a35fa1156db3640555f95ebb94f60a444e64d1c69626b0edef5df78eba225 \\\n    --hash=sha256:74ad5c3dad54a4641b4c28cd15ded70899d04459c6c7aeacafea716be97cce6d \\\n    --hash=sha256:74ea337e0ec3f6f342a36a4f1b5cd94dd9affddcd28ba9aae2905af932ee8c6b \\\n    --hash=sha256:75d9b1cf8258462dbdc1eeda718c96ea7f079324c09067f6daabfcf37712b7fe \\\n    --hash=sha256:77a4c8187a5948d7f8795adb765a3c7b553d07d86d88e43038fc32fc1fb9a3f3 \\\n    --hash=sha256:7824b5e8bdbf0bccb4ccd37bbb115849a1dc45437fb4de8351385ed07c437ee0 \\\n    --hash=sha256:7d38b0c279c3032e8c9cc013b405c6df8e1668dbf15465779aa7f15f61201812 \\\n    --hash=sha256:7e9c01d3dd7ceba4d1d436cc021d40d592466e40b9bc7f5d83dc4e98a5c9cd8c \\\n    --hash=sha256:7fd82debf43c6acd0a94359d232f6bb516ee13f269a7993736a9ac9f988bb5d9 \\\n    --hash=sha256:824c3d763a05ea9e9003610145186b0e9848c7584a5575c79bac5a8e7cd80bad \\\n    --hash=sha256:828f75af2b0080c8a972e75f649ab46af008e92c6104a57a759157200b835b75 \\\n    --hash=sha256:83f78128fa28705fa85d01c59771c72fe81c11bd0e6155edbb9f818983a7d761 \\\n    --hash=sha256:876bbfd276473d3daffe30e8c975df4ed9429967b41a6cb362dbb5155b6f13ad \\\n    --hash=sha256:886fc26012f0e8b5f69d1cfe6d711f6b11f194621539bf8e6bb1c25c5dc82724 \\\n    --hash=sha256:8a34616dc2521cc8dc1d7d081734da63539f021ac0450ce950908340c6e7aa2f \\\n    --hash=sha256:8a708a47ade1fe19e8371d5da076bac0dd4b0a5a7985ad6c637f7f7e361b6baa \\\n    --hash=sha256:8af9b142ad719ae3a911ebf616bc4b78b32bbab84d6a40d3ad2f129670509957 \\\n    --hash=sha256:8bf4df63592c2a66b4f8edc5df2544998c288aa02f96ce0acd880cd1de8c8127 \\\n    --hash=sha256:8de6f2a4ce7e7bd27d23dd94abf0ccafe0e0e5cc9c764b0577191f2c25f08f26 \\\n    --hash=sha256:8f8fddb8e323bd6eee4e54e69a39243beab22689070f4c66b472c4cc88bb89d8 \\\n    --hash=sha256:8fca690b00c4c48f6c2a547b0160ed511357093a4e4c9b47e0fadf3128066d89 \\\n    --hash=sha256:9506e892bcc3b409831d363c6f53e5985e1c8d1f6f6b0256d00358684ff85378 \\\n    --hash=sha256:958254518717542d02d0688d0d20cbf771da5e415e6f49543f92481c850a4540 \\\n    --hash=sha256:95a02752aa032eef4aed01cda6d9b687c669bd0396bf4519eef8bba22a286720 \\\n    --hash=sha256:96c30002424670b5e1e46495c2b8cbffef39cf77c1d79e76462029d50339785b \\\n    --hash=sha256:98b208a7cc42c803445ef551d6753cc42a5ea13e9cab1ee66cd8b9cb70195330 \\\n    --hash=sha256:9b3092d8992a1d69b7a59c3e39f35e1b9be327a17f68a7c35fc17329e337d6f2 \\\n    --hash=sha256:9e51c119992ea8820706871c30a4642ec76de20ae82f9b50b9a45517d8e9f810 \\\n    --hash=sha256:a5716a33bfabb2c6ce27b6cf03253467b3804f83e215f4d202685cf93c6c9874 \\\n    --hash=sha256:a5a00665d1a0e26763a7338d7e911d4598fbc1d50dd0d6b7919b7dc6c5d6569f \\\n    --hash=sha256:a5ca5aebae78a0bc13c1943af4af615d4966c5b650b05d5aa83b50e427196fee \\\n    --hash=sha256:a7b85b2cc6ea45e5f7e8c9a30bc9fabd47cda09106cbb4b967335c3e6c43b69d \\\n    --hash=sha256:a83ee7107df13abe42a54a6654670eef9bb39425cf2e27f65e0007465e1286ab \\\n    --hash=sha256:aa7d00b1700966d2917e54d278aba86897890ca9276dd8b76cf6446b6c181b92 \\\n    --hash=sha256:ab620eb663952455271ac37f9aaad86b73c969c02f11f53cea405b38e96a4300 \\\n    --hash=sha256:ad8b9671348d7c8716715652ae11f85ed0eb99e265a2df2ca490577d69860b2c \\\n    --hash=sha256:aefe930d113798330e9462f7874542977869c0613cba3262e2de3a8d5dee8f3a \\\n    --hash=sha256:b03af77d77e50edba2030fd5f7c352ff209314b09030a3cba7c14edf9a09a444 \\\n    --hash=sha256:b390aec180a7c054919c04898835e1c77bced23ea8383eb2c570213bf25d1a86 \\\n    --hash=sha256:b3d78f7bb2b9d9a30345be1474b9aaa8685430b54afb51ba3639b5c6c11e9ed6 \\\n    --hash=sha256:b5664603a253efd3a75716d793d1d3a6a82723b61dc6db767b2460bbbeec4c0f \\\n    --hash=sha256:b69602970994a2ed8bbfa78c2f0394a7435226c6040489702d9f0a0ad0c07052 \\\n    --hash=sha256:b6ae6a0328f0bc035741820fdeecdcd67bf4694eee03972e843663107122f450 \\\n    --hash=sha256:bad20d4c69c851c982a1e3606f4c293edfd5a87885786c50082412240c4b1ffd \\\n    --hash=sha256:bb7c99f0673c03017a3ee01e54a5c2617a05468b11eabe513b0080e063ed95b1 \\\n    --hash=sha256:bebb89489b279b2f5661bbbb2abcc87bcd4a46607bb4a5c966f04f1db6b8df9a \\\n    --hash=sha256:bfd1de989b3330420e29de39352f5c049905c9e3ee67233a50d550e3d652c148 \\\n    --hash=sha256:c2306e8bb53601979fcb3fa09cc65e031876d9ae01eff2fcbcd7a84ef94d5bc1 \\\n    --hash=sha256:c3a4e41e3096bf1f0f1b76e2ffd6d828d6547f574f702d59bdbef7acfa59db9c \\\n    --hash=sha256:c6834b92dd2428e2dd85ef3d85f723d3c12f20aaf43a2ddd4f944ca25d833408 \\\n    --hash=sha256:c90d3022d8a94778939cda8638c6c8da8fa757b8958dad7ec868ce29c87681b8 \\\n    --hash=sha256:ca307d6c259e5c98d3cb9ade55342b47a6839762caf2536f3d7b46ee660cc82e \\\n    --hash=sha256:ca7f6fe0f37ca978a1e5eb7a3a68e6413f417e78e838324947ffd420202b198b \\\n    --hash=sha256:cb6fae641357ed2f6e533c0d3c6504a4a5703621a50c89459e46051d56b61140 \\\n    --hash=sha256:cdaeeb6c350106df6bf9d873395973e5f066a9713200b72cd64f55d0a3eafab6 \\\n    --hash=sha256:cea20da04494e662b83c872683bf4ff2345206043d036315ed0e924b652e7294 \\\n    --hash=sha256:cea90547bfd93807e0013a004dc76552be44fad3bc1cc2b38610a9e889ed098f \\\n    --hash=sha256:d09037ca068d784ebc4aec290ef952ca27ac15dd9c0b5801a88c6e1096b83e6b \\\n    --hash=sha256:d27c2123977cb9269c30a49ba45f03a4323017ef693e19db4ec9dbe1299a3002 \\\n    --hash=sha256:d50de98e8d807dc31822fff96f50293163a62418eb65487a21b42713d72ed0b7 \\\n    --hash=sha256:d66a64dd5dec136040ec2ae94aa026a912ee60fdd45bc28d3db30037fd809e88 \\\n    --hash=sha256:d79308fa689fac89cbcfbd4dbfc80b5f95c54c5a7fd4d194be221f9d33d026e6 \\\n    --hash=sha256:da3275833be0edbaf4830fae08bae3dc7219f40ce0c37eaa6c25825957e06612 \\\n    --hash=sha256:dc1a26b8e53395a01c2c611e58602fa47461f136fba7cd5542e6db6d64be1839 \\\n    --hash=sha256:dc23390afe9f4ef9ac3bcc72a03a56eebbde03f4c571a32cb38f859cff9a6524 \\\n    --hash=sha256:e05c2f7925f1d88778e53cb44f14e0223204a3bdd09a41664750363acfb1f2ef \\\n    --hash=sha256:e12dfea7f5fc2a34a9080efbf79c4c44eb380ec5b9c6fea09407e08f0d1e941d \\\n    --hash=sha256:e4e4523d6f336708d732516e6cfca7796cf3d96c9474eb5aecf6165f2f1fefc3 \\\n    --hash=sha256:e4e49f7e1a4e7191bdf9dc67a974db714501b1fc52c24324103d06a86abd5c08 \\\n    --hash=sha256:e68e151428b5384f766cd25739bf77c7e4a3dc93b5ded7a12118d9fbfdf78ab6 \\\n    --hash=sha256:e8e4d953faaded9ec7ede36824e9814082d22d4c7b1eafbfa079ecba8cd0d076 \\\n    --hash=sha256:ee9df1f0d77b9c6e94f4ac0fec533fbddd5ea3a327807f18d7b069ae019ded80 \\\n    --hash=sha256:f0a887b6565bbfe80efde2b7f6e8890d7d9bbdb11bdb17028a3690c32fe0621f \\\n    --hash=sha256:f0f4a42db92d6ec7677ab9d12830a2a8ec145a9c6d15db2b593466bc875c78d7 \\\n    --hash=sha256:f1303ef2eec81262a4b708c3e858afe58d7c75ad91c1c05266eda7673369859a \\\n    --hash=sha256:f1d56ec54d257d05e0b50f5780d967540cd07beeaf9e5f645b26d50cce79f4d8 \\\n    --hash=sha256:f4167e87b397f273dc2356fcf1eaf50a6bac51e6105f45103ef7129c8efb0255 \\\n    --hash=sha256:f76fc85bd054c806960f917ec0f329e24e436f1712267d90588e4c39890caa63 \\\n    --hash=sha256:f942903fde7363d1d879057ec5de01310efda2597161784d752fa9953a01a71a \\\n    --hash=sha256:f9b1c4900736e489a812c529100de4b8fb617d4db075e931e213c57424b83d9b \\\n    --hash=sha256:fc271a6f0a2126958f4090e5507b9da5848927dae331f8f763bd4aa642b3d2cd \\\n    --hash=sha256:febcce10f2bcdbb80b4ea919238a6a4ac13dbc4c7cadbe8d5d75c3682f8b5404\n    # via matplotlib\nmarkupsafe==3.0.4 \\\n    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \\\n    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \\\n    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \\\n    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \\\n    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \\\n    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \\\n    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \\\n    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \\\n    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \\\n    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \\\n    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \\\n    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \\\n    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \\\n    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \\\n    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \\\n    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \\\n    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \\\n    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \\\n    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \\\n    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \\\n    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \\\n    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \\\n    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \\\n    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \\\n    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \\\n    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \\\n    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \\\n    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \\\n    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \\\n    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \\\n    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \\\n    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \\\n    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \\\n    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \\\n    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \\\n    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \\\n    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \\\n    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \\\n    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \\\n    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \\\n    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \\\n    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \\\n    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \\\n    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \\\n    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \\\n    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \\\n    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \\\n    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \\\n    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \\\n    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \\\n    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \\\n    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \\\n    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \\\n    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \\\n    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \\\n    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \\\n    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \\\n    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \\\n    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \\\n    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \\\n    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \\\n    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \\\n    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \\\n    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \\\n    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \\\n    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \\\n    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \\\n    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \\\n    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \\\n    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \\\n    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \\\n    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \\\n    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \\\n    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \\\n    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \\\n    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \\\n    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \\\n    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \\\n    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \\\n    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \\\n    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \\\n    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \\\n    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \\\n    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \\\n    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \\\n    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \\\n    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \\\n    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \\\n    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \\\n    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \\\n    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \\\n    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \\\n    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \\\n    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \\\n    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \\\n    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \\\n    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \\\n    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \\\n    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \\\n    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \\\n    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \\\n    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \\\n    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \\\n    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \\\n    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \\\n    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \\\n    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \\\n    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \\\n    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \\\n    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \\\n    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \\\n    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \\\n    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \\\n    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \\\n    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \\\n    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \\\n    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \\\n    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \\\n    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \\\n    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \\\n    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \\\n    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \\\n    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \\\n    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \\\n    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \\\n    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \\\n    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \\\n    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \\\n    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \\\n    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \\\n    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \\\n    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \\\n    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \\\n    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \\\n    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \\\n    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \\\n    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \\\n    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \\\n    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \\\n    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \\\n    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \\\n    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \\\n    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \\\n    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \\\n    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \\\n    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \\\n    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \\\n    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \\\n    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \\\n    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \\\n    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3\n    # via jinja2\nmatplotlib==3.11.2 \\\n    --hash=sha256:01dc8eaaab5a9fce9ff615eca82345728f289e4715b186ee10c6d85272fc26bb \\\n    --hash=sha256:02329432ae5c6af87cf208ee575b701d698bdf0b1a3bb28cc6d53c36e967e575 \\\n    --hash=sha256:07d9b9fa60cd4c393692f50d0bb03123242ddf61c99bb0e95e75feb354e7c1a8 \\\n    --hash=sha256:116cdb0eb0eb5644fc98eb2975d4b4dd4ad35e5c8e6b851c22e6976f371f7ab5 \\\n    --hash=sha256:1944895967f87c84c9b4bad29a707b31f5b36df4a3a2339ea1f8ea3ce5105539 \\\n    --hash=sha256:1a3040b209f3968b4e84161df7b174f07a9fad33b0f2d7e48ea3bbd3075e2863 \\\n    --hash=sha256:1b9a7ad579856284135e401ecc918c5f8a017ee30539298862a109f51b971710 \\\n    --hash=sha256:254d4ddb2fa8df3b4c689c0c306063aee10521df82cfb438185e499c75fe37c1 \\\n    --hash=sha256:2a8285cea8ef4d92aa041d1c33788bcae82248503300f93f1ca2136b9049452f \\\n    --hash=sha256:30ec15d7eefee71de16b7c689b42ba49715a4644650b76a6c7c70d79daf24e91 \\\n    --hash=sha256:399fef672f7046ef7d6a57572b2a6f9845f3f5afcff04e7b2df7a363a9f42190 \\\n    --hash=sha256:3aa4b8516fd26659e4363abbf317c703d9116496c5db2e9d0609a2866dd39dd2 \\\n    --hash=sha256:3da3bc0cbf7245e7db72cc6d29d12c5abef72cb73059c73b945f14e3545f3eb2 \\\n    --hash=sha256:3e8576f7c47e02fd4f21f44171302d2d1d58d4471d46da3d71fe8899d19539d9 \\\n    --hash=sha256:57b9ea60a835937c2012861923cbb91f47db8565775d326d1c42fc926aa10351 \\\n    --hash=sha256:5e1e923a3fc3326b99ec0a6ff1ab1338ac6c6cc62ad9d8a9c944197c7f8c6221 \\\n    --hash=sha256:643ff850d8e0f5b8319337f87ed3cb59506afb3df3cc48de777d85871233be7b \\\n    --hash=sha256:75b6d88402770e181b5d06a67dda4129c31da7d05004d63a21c310ec78d1b83c \\\n    --hash=sha256:79a258f58253dfa025af80a9e9bb228d75fced007f0e93ae7423fefbde81a74d \\\n    --hash=sha256:7d43ff8cebb50840648cb6429b2228621dbd709010f3117b3740abb20abf21c0 \\\n    --hash=sha256:7e5a90f8a707ebb6004a713b1cff091a40cc5df4c7c1cb165e5a505ebc11c292 \\\n    --hash=sha256:7ef7a53b66780e5d942923724f08577fbc5be1f7322da0f0f3f9dcaa45dd803d \\\n    --hash=sha256:854df8d7dfe9fdffcbaa6f39e44a6c24b409cb4d7561fbc09213b157d833f6a6 \\\n    --hash=sha256:894a9cbbecbe30ae6787d464df2e8fc7a8d475cfc68f87c3029f7c11152899b3 \\\n    --hash=sha256:8c8255de28f986d935a64c9ca71c0ec2d2f41d355691f5ea684725dc91413f71 \\\n    --hash=sha256:930efb28f59fda124e39265d177bab302625297bb147d2910de725a2fc2aef54 \\\n    --hash=sha256:a24d5fd36e4f0e742c3851dcd20810e56a95633a342e4bf6cb591c678e8fe61f \\\n    --hash=sha256:a6939df7567114b6bac7f4c5e06c84a67f197c1b2f2e4b234d4eecb3bfec9482 \\\n    --hash=sha256:a8756cc73d9af9a7fe0deb54ea2e75ef73b01d9e575877e72acad5458e660943 \\\n    --hash=sha256:af2661f6ac6bbd1d081996f54fdd9385715c625ace8cec0869055c0cfbf38981 \\\n    --hash=sha256:bbf1062991d826ed27e2144f3afa4461afbb8ff56e8f703043e191a8163b1ee9 \\\n    --hash=sha256:bc067c462a86f0e57bf52fc6e058d90171a5420007dac00c46e90153050c69a7 \\\n    --hash=sha256:bf3fe71fbfb8ec0e310e0bc8537c3405a01f38f25f9394ed2135e6202fed542b \\\n    --hash=sha256:c0b83f044ce10a98027b105b3931548719a6e8c7ef986b4362651e0b5367c8dc \\\n    --hash=sha256:c27e577ece613ea12a0790e00b4eb80d901c59a3e16cad474f31d8b1529690b9 \\\n    --hash=sha256:c5c1c68ee401fc98271263410f0e5ce88285abacf7627132914e8adf3d70ff43 \\\n    --hash=sha256:c9721f81275499da1feeb36a2cf8192ea086283b3bd16b7dc4c9d7aedb7396d6 \\\n    --hash=sha256:cc82dde2a0d3e3ad472edce04897ad7146b8d8bfd1df8a32992eebb81af18fdc \\\n    --hash=sha256:cec596316640f2b394b8f0daa0ea61a8eae82d017b620b9f202befb972a59ea4 \\\n    --hash=sha256:cf41ecd1b0c0b6f7177ed965a54c2afbe888715c7cf6054dc12d53bc1494002c \\\n    --hash=sha256:d3304eb5a59442a8867f6920d484591c0fa09ffc29e9260be2feec3351e25869 \\\n    --hash=sha256:d480038c83691532ed52ff3147db51fa902fc78cb2d8349993a1cdb684435bff \\\n    --hash=sha256:df4f7784aca81a94f254c0a2767d592ee25f407e488f5fa7203e51093fb6ca27 \\\n    --hash=sha256:e43b188f0a5b75447bcc197728258166aa64365770ed1caa36595a5e1ca4bbba \\\n    --hash=sha256:e60cf3047a51edecdc4535a9196bbd6a732936b8e9f8184aabf4d16165884aaf \\\n    --hash=sha256:eac4b07d4e3743b172451e122ea964f72f152879d4f9adcf3f3d33e518f12ead \\\n    --hash=sha256:eb3712dc9b464793de0a4e42a7313d50293c751f94bddaf7332a1bc71bccdda9 \\\n    --hash=sha256:ecea603dd2fbf8242fd31a305a8b12a4ece2de28096870c65fdd0d1e35b8d9a6 \\\n    --hash=sha256:ef31985c4dedb5f1424e1aec6849a47dd37689cb7fa3c20b1b82187f26806261 \\\n    --hash=sha256:ef752769cd962f39ea0b6ffc82d1ea43a0012c5a6157c7a075212fa509cfcff2 \\\n    --hash=sha256:f25446b2981717dca9786bac841cb3fd7efb568e3e3c755dd980481c5cb9228d \\\n    --hash=sha256:f2ac30cf5eb5dff1b584627ae0b0e1186551a4f69ae3c75073911da497a29170 \\\n    --hash=sha256:fea03cf56568cc1cba08b470be6a0559e71c3a5b688d54b7179bb35ba23d0821\n    # via mlstm-kernels\nmlstm-kernels==2.0.6 \\\n    --hash=sha256:85eeea2c1867a2fbd451ec2ddd6065a9d29d3a3022b8bb4ffe02c20e27c56df8 \\\n    --hash=sha256:9e1a3866dc4db964a27d833e6c17fc0da7d74a2ed6aa9d4daf2cbdfbd42a9584\n    # via xlstm\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n    # via sympy\nnetworkx==3.7 \\\n    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n    # via torch\nninja==1.13.2 \\\n    --hash=sha256:09de9ab04f7352f51570c73fd4913acb1e6c24be0a72cd8b80243d4d3ed04925 \\\n    --hash=sha256:0e083700470c02ca154a855ae6d692d03564f5064cae52e113896f9ccc078418 \\\n    --hash=sha256:1293f4078278b70d0ee4b6cc8f3a9e030656c9b2f59909970343c4fe76070118 \\\n    --hash=sha256:1684c60d031c54c1d049541b64243c0c567dca5463dbd77682a8901780af293d \\\n    --hash=sha256:1db9852e528efa7702f5123969f86678663e46d57ff28ab13f5fd84d64a85fb1 \\\n    --hash=sha256:227cbc3ae3e5e429692388103cae8c09451df086cd2d342dae0795af0d162547 \\\n    --hash=sha256:525bfa3fc88aa30a4467df270fd5be6f9fcae8061d54d4df74ea1dc5abd5a975 \\\n    --hash=sha256:59d71c3e15b6b6f3d903eb0c27285544e0747ca59925ada7037bb1af781ad4b3 \\\n    --hash=sha256:65a24341b5ac09fcadcc37082660be40a94174e51a937fabf6e2cae26225fa2c \\\n    --hash=sha256:6a87bf42b123abe2f37737300185f0a303a891899da85d73a3613ee80547e578 \\\n    --hash=sha256:792cadbb9decfd1f776d4d0a6930feb46d08302eb57c176bcf26b09de5748e9f \\\n    --hash=sha256:81d95081c0ad7c95f67bf682220361ed2a32659d7861b4766b03433c58f22516 \\\n    --hash=sha256:915bd482c4be41c75120fd67a22e0bb3f0fbb3bbc5f95b89787deadd59e27ef2 \\\n    --hash=sha256:919572cbc3f233261ecd41fe1f3efc9d44aa02464a4588867e06a8b4f6f416ea \\\n    --hash=sha256:aa3d2ae5706a2c4d1e93edc951d1c6cbb45107413c404f8fde1741239efbc9a0 \\\n    --hash=sha256:b2f687437fac460b27b7eadc99039b1163016fb4ba7276e2782a192d9f24ee0e \\\n    --hash=sha256:d775a5e43e9088f507a6250d57fcf5678eb31268c545feb5064ffeee33735622 \\\n    --hash=sha256:f90f84affc441e219f15fe52532806c1c9dbd22fb66c3addddce88a3deaabab7 \\\n    --hash=sha256:fd82e26c0706ad4ab88e5fdd26f3fab0a987a90f810160f6c322e752c6af298b\n    # via\n    #   flashrnn\n    #   xlstm\nnumpy==2.3.3 \\\n    --hash=sha256:067e3d7159a5d8f8a0b46ee11148fc35ca9b21f61e3c49fbd0a027450e65a33b \\\n    --hash=sha256:0edd58682a399824633b66885d699d7de982800053acf20be1eaa46d92009c54 \\\n    --hash=sha256:0ffc4f5caba7dfcbe944ed674b7eef683c7e94874046454bb79ed7ee0236f59d \\\n    --hash=sha256:1250c5d3d2562ec4174bce2e3a1523041595f9b651065e4a4473f5f48a6bc8a5 \\\n    --hash=sha256:179a42101b845a816d464b6fe9a845dfaf308fdfc7925387195570789bb2c970 \\\n    --hash=sha256:1c02d0629d25d426585fb2e45a66154081b9fa677bc92a881ff1d216bc9919a8 \\\n    --hash=sha256:1e02c7159791cd481e1e6d5ddd766b62a4d5acf8df4d4d1afe35ee9c5c33a41e \\\n    --hash=sha256:2990adf06d1ecee3b3dcbb4977dfab6e9f09807598d647f04d385d29e7a3c3d3 \\\n    --hash=sha256:2e267c7da5bf7309670523896df97f93f6e469fb931161f483cd6882b3b1a5dc \\\n    --hash=sha256:367ad5d8fbec5d9296d18478804a530f1191e24ab4d75ab408346ae88045d25e \\\n    --hash=sha256:396b254daeb0a57b1fe0ecb5e3cff6fa79a380fa97c8f7781a6d08cd429418fe \\\n    --hash=sha256:3c7cf302ac6e0b76a64c4aecf1a09e51abd9b01fc7feee80f6c43e3ab1b1dbc5 \\\n    --hash=sha256:40051003e03db4041aa325da2a0971ba41cf65714e65d296397cc0e32de6018b \\\n    --hash=sha256:414a97499480067d305fcac9716c29cf4d0d76db6ebf0bf3cbce666677f12652 \\\n    --hash=sha256:433bf137e338677cebdd5beac0199ac84712ad9d630b74eceeb759eaa45ddf30 \\\n    --hash=sha256:4384a169c4d8f97195980815d6fcad04933a7e1ab3b530921c3fef7a1c63426d \\\n    --hash=sha256:497d7cad08e7092dba36e3d296fe4c97708c93daf26643a1ae4b03f6294d30eb \\\n    --hash=sha256:50a5fe69f135f88a2be9b6ca0481a68a136f6febe1916e4920e12f1a34e708a7 \\\n    --hash=sha256:533ca5f6d325c80b6007d4d7fb1984c303553534191024ec6a524a4c92a5935a \\\n    --hash=sha256:5534ed6b92f9b7dca6c0a19d6df12d41c68b991cef051d108f6dbff3babc4ebf \\\n    --hash=sha256:5b83648633d46f77039c29078751f80da65aa64d5622a3cd62aaef9d835b6c93 \\\n    --hash=sha256:691808c2b26b0f002a032c73255d0bd89751425f379f7bcd22d140db593a96e8 \\\n    --hash=sha256:6ee9086235dd6ab7ae75aba5662f582a81ced49f0f1c6de4260a78d8f2d91a19 \\\n    --hash=sha256:74c2a948d02f88c11a3c075d9733f1ae67d97c6bdb97f2bb542f980458b257e7 \\\n    --hash=sha256:75370986cc0bc66f4ce5110ad35aae6d182cc4ce6433c40ad151f53690130bf1 \\\n    --hash=sha256:78c9f6560dc7e6b3990e32df7ea1a50bbd0e2a111e05209963f5ddcab7073b0b \\\n    --hash=sha256:7af05ed4dc19f308e1d9fc759f36f21921eb7bbfc82843eeec6b2a2863a0aefa \\\n    --hash=sha256:7f025652034199c301049296b59fa7d52c7e625017cae4c75d8662e377bf487d \\\n    --hash=sha256:823d04112bc85ef5c4fda73ba24e6096c8f869931405a80aa8b0e604510a26bc \\\n    --hash=sha256:8596ba2f8af5f93b01d97563832686d20206d303024777f6dfc2e7c7c3f1850e \\\n    --hash=sha256:8e9aced64054739037d42fb84c54dd38b81ee238816c948c8f3ed134665dcd86 \\\n    --hash=sha256:8f6ac61a217437946a1fa48d24c47c91a0c4f725237871117dea264982128097 \\\n    --hash=sha256:901bf6123879b7f251d3631967fd574690734236075082078e0571977c6a8e6a \\\n    --hash=sha256:93d4962d8f82af58f0b2eb85daaf1b3ca23fe0a85d0be8f1f2b7bb46034e56d7 \\\n    --hash=sha256:94fcaa68757c3e2e668ddadeaa86ab05499a70725811e582b6a9858dd472fb30 \\\n    --hash=sha256:952cfd0748514ea7c3afc729a0fc639e61655ce4c55ab9acfab14bda4f402b4c \\\n    --hash=sha256:9591e1221db3f37751e6442850429b3aabf7026d3b05542d102944ca7f00c8a8 \\\n    --hash=sha256:99683cbe0658f8271b333a1b1b4bb3173750ad59c0c61f5bbdc5b318918fffe3 \\\n    --hash=sha256:9ad12e976ca7b10f1774b03615a2a4bab8addce37ecc77394d8e986927dc0dfe \\\n    --hash=sha256:9cc48e09feb11e1db00b320e9d30a4151f7369afb96bd0e48d942d09da3a0d00 \\\n    --hash=sha256:9dc13c6a5829610cc07422bc74d3ac083bd8323f14e2827d992f9e52e22cd6a6 \\\n    --hash=sha256:9e318ee0596d76d4cb3d78535dc005fa60e5ea348cd131a51e99d0bdbe0b54fe \\\n    --hash=sha256:a333b4ed33d8dc2b373cc955ca57babc00cd6f9009991d9edc5ddbc1bac36bcd \\\n    --hash=sha256:afd07d377f478344ec6ca2b8d4ca08ae8bd44706763d1efb56397de606393f48 \\\n    --hash=sha256:b001bae8cea1c7dfdb2ae2b017ed0a6f2102d7a70059df1e338e307a4c78a8ae \\\n    --hash=sha256:b37a0b2e5935409daebe82c1e42274d30d9dd355852529eab91dab8dcca7419f \\\n    --hash=sha256:b912f2ed2b67a129e6a601e9d93d4fa37bef67e54cac442a2f588a54afe5c67a \\\n    --hash=sha256:bc92a5dedcc53857249ca51ef29f5e5f2f8c513e22cfb90faeb20343b8c6f7a6 \\\n    --hash=sha256:ca0309a18d4dfea6fc6262a66d06c26cfe4640c3926ceec90e57791a82b6eee5 \\\n    --hash=sha256:cb248499b0bc3be66ebd6578b83e5acacf1d6cb2a77f2248ce0e40fbec5a76d0 \\\n    --hash=sha256:cb32e3cf0f762aee47ad1ddc6672988f7f27045b0783c887190545baba73aa25 \\\n    --hash=sha256:cd052f1fa6a78dee696b58a914b7229ecfa41f0a6d96dc663c1220a55e137593 \\\n    --hash=sha256:cd4260f64bc794c3390a63bf0728220dd1a68170c169088a1e0dfa2fde1be12f \\\n    --hash=sha256:cd7de500a5b66319db419dc3c345244404a164beae0d0937283b907d8152e6ea \\\n    --hash=sha256:ce020080e4a52426202bdb6f7691c65bb55e49f261f31a8f506c9f6bc7450421 \\\n    --hash=sha256:cfdd09f9c84a1a934cde1eec2267f0a43a7cd44b2cca4ff95b7c0d14d144b0bf \\\n    --hash=sha256:d00de139a3324e26ed5b95870ce63be7ec7352171bc69a4cf1f157a48e3eb6b7 \\\n    --hash=sha256:d79715d95f1894771eb4e60fb23f065663b2298f7d22945d66877aadf33d00c7 \\\n    --hash=sha256:d8f3b1080782469fdc1718c4ed1d22549b5fb12af0d57d35e992158a772a37cf \\\n    --hash=sha256:d9192da52b9745f7f0766531dcfa978b7763916f158bb63bdb8a1eca0068ab20 \\\n    --hash=sha256:d9d537a39cc9de668e5cd0e25affb17aec17b577c6b3ae8a3d866b479fbe88d0 \\\n    --hash=sha256:da1a74b90e7483d6ce5244053399a614b1d6b7bc30a60d2f570e5071f8959d3e \\\n    --hash=sha256:dca2d0fc80b3893ae72197b39f69d55a3cd8b17ea1b50aa4c62de82419936150 \\\n    --hash=sha256:ddc7c39727ba62b80dfdbedf400d1c10ddfa8eefbd7ec8dcb118be8b56d31029 \\\n    --hash=sha256:e1ec5615b05369925bd1125f27df33f3b6c8bc10d788d5999ecd8769a1fa04db \\\n    --hash=sha256:e6687dc183aa55dae4a705b35f9c0f8cb178bcaa2f029b241ac5356221d5c021 \\\n    --hash=sha256:e7e946c7170858a0295f79a60214424caac2ffdb0063d4d79cb681f9aa0aa569 \\\n    --hash=sha256:eb63d443d7b4ffd1e873f8155260d7f58e7e4b095961b01c91062935c2491e57 \\\n    --hash=sha256:ec9d249840f6a565f58d8f913bccac2444235025bbb13e9a4681783572ee3caa \\\n    --hash=sha256:ed635ff692483b8e3f0fcaa8e7eb8a75ee71aa6d975388224f70821421800cea \\\n    --hash=sha256:eda59e44957d272846bb407aad19f89dc6f58fecf3504bd144f4c5cf81a7eacc \\\n    --hash=sha256:f0dadeb302887f07431910f67a14d57209ed91130be0adea2f9793f1a4f817cf \\\n    --hash=sha256:f0ddb4b96a87b6728df9362135e764eac3cfa674499943ebc44ce96c478ab125 \\\n    --hash=sha256:f5415fb78995644253370985342cd03572ef8620b934da27d77377a2285955bf\n    # via\n    #   tirex-forecasting-pipeline (pyproject.toml)\n    #   contourpy\n    #   matplotlib\n    #   mlstm-kernels\n    #   pandas\n    #   tirex-2\n    #   torchvision\nnvidia-cublas-cu12==12.8.4.1 \\\n    --hash=sha256:47e9b82132fa8d2b4944e708049229601448aaad7e6f296f630f2d1a32de35af \\\n    --hash=sha256:8ac4e771d5a348c551b2a426eda6193c19aa630236b418086020df5ba9667142 \\\n    --hash=sha256:b86f6dd8935884615a0683b663891d43781b819ac4f2ba2b0c9604676af346d0\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.8.90 \\\n    --hash=sha256:4412396548808ddfed3f17a467b104ba7751e6b58678a4b840675c56d21cf7ed \\\n    --hash=sha256:bb479dcdf7e6d4f8b0b01b115260399bf34154a1a2e9fe11c85c517d87efd98e \\\n    --hash=sha256:ea0cb07ebda26bb9b29ba82cda34849e73c166c18162d3913575b0c9db9a6182\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.8.93 \\\n    --hash=sha256:7a4b6b2904850fe78e0bd179c4b655c404d4bb799ef03ddc60804247099ae909 \\\n    --hash=sha256:a7756528852ef889772a84c6cd89d41dfa74667e24cca16bb31f8f061e3e9994 \\\n    --hash=sha256:fc1fec1e1637854b4c0a65fb9a8346b51dd9ee69e61ebaccc82058441f15bce8\n    # via torch\nnvidia-cuda-runtime-cu12==12.8.90 \\\n    --hash=sha256:52bf7bbee900262ffefe5e9d5a2a69a30d97e2bc5bb6cc866688caa976966e3d \\\n    --hash=sha256:adade8dcbd0edf427b7204d480d6066d33902cab2a4707dcfc48a2d0fd44ab90 \\\n    --hash=sha256:c0c6027f01505bfed6c3b21ec546f69c687689aad5f1a377554bc6ca4aa993a8\n    # via torch\nnvidia-cudnn-cu12==9.10.2.21 \\\n    --hash=sha256:949452be657fa16687d0930933f032835951ef0892b37d2d53824d1a84dc97a8 \\\n    --hash=sha256:c6288de7d63e6cf62988f0923f96dc339cea362decb1bf5b3141883392a7d65e \\\n    --hash=sha256:c9132cc3f8958447b4910a1720036d9eff5928cc3179b0a51fb6d167c6cc87d8\n    # via torch\nnvidia-cufft-cu12==11.3.3.83 \\\n    --hash=sha256:4d2dd21ec0b88cf61b62e6b43564355e5222e4a3fb394cac0db101f2dd0d4f74 \\\n    --hash=sha256:7a64a98ef2a7c47f905aaf8931b69a3a43f27c55530c698bb2ed7c75c0b42cb7 \\\n    --hash=sha256:848ef7224d6305cdb2a4df928759dca7b1201874787083b6e7550dd6765ce69a\n    # via torch\nnvidia-cufile-cu12==1.13.1.3 \\\n    --hash=sha256:1d069003be650e131b21c932ec3d8969c1715379251f8d23a1860554b1cb24fc \\\n    --hash=sha256:4beb6d4cce47c1a0f1013d72e02b0994730359e17801d395bdcbf20cfb3bb00a\n    # via torch\nnvidia-curand-cu12==10.3.9.90 \\\n    --hash=sha256:b32331d4f4df5d6eefa0554c565b626c7216f87a06a4f56fab27c3b68a830ec9 \\\n    --hash=sha256:dfab99248034673b779bc6decafdc3404a8a6f502462201f2f31f11354204acd \\\n    --hash=sha256:f149a8ca457277da854f89cf282d6ef43176861926c7ac85b2a0fbd237c587ec\n    # via torch\nnvidia-cusolver-cu12==11.7.3.90 \\\n    --hash=sha256:4376c11ad263152bd50ea295c05370360776f8c3427b30991df774f9fb26c450 \\\n    --hash=sha256:4a550db115fcabc4d495eb7d39ac8b58d4ab5d8e63274d3754df1c0ad6a22d34 \\\n    --hash=sha256:db9ed69dbef9715071232caa9b69c52ac7de3a95773c2db65bdba85916e4e5c0\n    # via torch\nnvidia-cusparse-cu12==12.5.8.93 \\\n    --hash=sha256:1ec05d76bbbd8b61b06a80e1eaf8cf4959c3d4ce8e711b65ebd0443bb0ebb13b \\\n    --hash=sha256:9a33604331cb2cac199f2e7f5104dfbb8a5a898c367a53dfda9ff2acb6b6b4dd \\\n    --hash=sha256:9b6c161cb130be1a07a27ea6923df8141f3c295852f4b260c65f18f3e0a091dc\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.7.1 \\\n    --hash=sha256:8878dce784d0fac90131b6817b607e803c36e629ba34dc5b433471382196b6a5 \\\n    --hash=sha256:f1bb701d6b930d5a7cea44c19ceb973311500847f81b634d802b7b539dc55623 \\\n    --hash=sha256:f67fbb5831940ec829c9117b7f33807db9f9678dc2a617fbe781cac17b4e1075\n    # via torch\nnvidia-nccl-cu12==2.27.3 \\\n    --hash=sha256:9ddf1a245abc36c550870f26d537a9b6087fb2e2e3d6e0ef03374c6fd19d984f \\\n    --hash=sha256:adf27ccf4238253e0b826bce3ff5fa532d65fc42322c8bfdfaf28024c0fbe039\n    # via torch\nnvidia-nvjitlink-cu12==12.8.93 \\\n    --hash=sha256:81ff63371a7ebd6e6451970684f916be2eab07321b73c9d244dc2b4da7f73b88 \\\n    --hash=sha256:adccd7161ace7261e01bb91e44e88da350895c270d23f744f0820c818b7229e7 \\\n    --hash=sha256:bd93fbeeee850917903583587f4fc3a4eafa022e34572251368238ab5e6bd67f\n    # via\n    #   nvidia-cufft-cu12\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.8.90 \\\n    --hash=sha256:5b17e2001cc0d751a5bc2c6ec6d26ad95913324a4adb86788c944f8ce9ba441f \\\n    --hash=sha256:619c8304aedc69f02ea82dd244541a83c3d9d40993381b3b590f1adaed3db41e \\\n    --hash=sha256:d7ad891da111ebafbf7e015d34879f7112832fc239ff0d7d776b6cb685274615\n    # via torch\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   huggingface-hub\n    #   matplotlib\n    #   xlstm\npandas==2.3.3 \\\n    --hash=sha256:0242fe9a49aa8b4d78a4fa03acb397a58833ef6199e9aa40a95f027bb3a1b6e7 \\\n    --hash=sha256:1611aedd912e1ff81ff41c745822980c49ce4a7907537be8692c8dbc31924593 \\\n    --hash=sha256:1b07204a219b3b7350abaae088f451860223a52cfb8a6c53358e7948735158e5 \\\n    --hash=sha256:1d37b5848ba49824e5c30bedb9c830ab9b7751fd049bc7914533e01c65f79791 \\\n    --hash=sha256:23ebd657a4d38268c7dfbdf089fbc31ea709d82e4923c5ffd4fbd5747133ce73 \\\n    --hash=sha256:2462b1a365b6109d275250baaae7b760fd25c726aaca0054649286bcfbb3e8ec \\\n    --hash=sha256:28083c648d9a99a5dd035ec125d42439c6c1c525098c58af0fc38dd1a7a1b3d4 \\\n    --hash=sha256:2e3ebdb170b5ef78f19bfb71b0dc5dc58775032361fa188e814959b74d726dd5 \\\n    --hash=sha256:318d77e0e42a628c04dc56bcef4b40de67918f7041c2b061af1da41dcff670ac \\\n    --hash=sha256:371a4ab48e950033bcf52b6527eccb564f52dc826c02afd9a1bc0ab731bba084 \\\n    --hash=sha256:376c6446ae31770764215a6c937f72d917f214b43560603cd60da6408f183b6c \\\n    --hash=sha256:3869faf4bd07b3b66a9f462417d0ca3a9df29a9f6abd5d0d0dbab15dac7abe87 \\\n    --hash=sha256:3fd2f887589c7aa868e02632612ba39acb0b8948faf5cc58f0850e165bd46f35 \\\n    --hash=sha256:4793891684806ae50d1288c9bae9330293ab4e083ccd1c5e383c34549c6e4250 \\\n    --hash=sha256:4e0a175408804d566144e170d0476b15d78458795bb18f1304fb94160cabf40c \\\n    --hash=sha256:503cf027cf9940d2ceaa1a93cfb5f8c8c7e6e90720a2850378f0b3f3b1e06826 \\\n    --hash=sha256:5554c929ccc317d41a5e3d1234f3be588248e61f08a74dd17c9eabb535777dc9 \\\n    --hash=sha256:56851a737e3470de7fa88e6131f41281ed440d29a9268dcbf0002da5ac366713 \\\n    --hash=sha256:5caf26f64126b6c7aec964f74266f435afef1c1b13da3b0636c7518a1fa3e2b1 \\\n    --hash=sha256:602b8615ebcc4a0c1751e71840428ddebeb142ec02c786e8ad6b1ce3c8dec523 \\\n    --hash=sha256:6253c72c6a1d990a410bc7de641d34053364ef8bcd3126f7e7450125887dffe3 \\\n    --hash=sha256:6435cb949cb34ec11cc9860246ccb2fdc9ecd742c12d3304989017d53f039a78 \\\n    --hash=sha256:6d21f6d74eb1725c2efaa71a2bfc661a0689579b58e9c0ca58a739ff0b002b53 \\\n    --hash=sha256:6d2cefc361461662ac48810cb14365a365ce864afe85ef1f447ff5a1e99ea81c \\\n    --hash=sha256:74ecdf1d301e812db96a465a525952f4dde225fdb6d8e5a521d47e1f42041e21 \\\n    --hash=sha256:75ea25f9529fdec2d2e93a42c523962261e567d250b0013b16210e1d40d7c2e5 \\\n    --hash=sha256:854d00d556406bffe66a4c0802f334c9ad5a96b4f1f868adf036a21b11ef13ff \\\n    --hash=sha256:8fe25fc7b623b0ef6b5009149627e34d2a4657e880948ec3c840e9402e5c1b45 \\\n    --hash=sha256:900f47d8f20860de523a1ac881c4c36d65efcb2eb850e6948140fa781736e110 \\\n    --hash=sha256:93c2d9ab0fc11822b5eece72ec9587e172f63cff87c00b062f6e37448ced4493 \\\n    --hash=sha256:a16dcec078a01eeef8ee61bf64074b4e524a2a3f4b3be9326420cabe59c4778b \\\n    --hash=sha256:a21d830e78df0a515db2b3d2f5570610f5e6bd2e27749770e8bb7b524b89b450 \\\n    --hash=sha256:a45c765238e2ed7d7c608fc5bc4a6f88b642f2f01e70c0c23d2224dd21829d86 \\\n    --hash=sha256:a637c5cdfa04b6d6e2ecedcb81fc52ffb0fd78ce2ebccc9ea964df9f658de8c8 \\\n    --hash=sha256:a68e15f780eddf2b07d242e17a04aa187a7ee12b40b930bfdd78070556550e98 \\\n    --hash=sha256:b3d11d2fda7eb164ef27ffc14b4fcab16a80e1ce67e9f57e19ec0afaf715ba89 \\\n    --hash=sha256:b468d3dad6ff947df92dcb32ede5b7bd41a9b3cceef0a30ed925f6d01fb8fa66 \\\n    --hash=sha256:b98560e98cb334799c0b07ca7967ac361a47326e9b4e5a7dfb5ab2b1c9d35a1b \\\n    --hash=sha256:bdcd9d1167f4885211e401b3036c0c8d9e274eee67ea8d0758a256d60704cfe8 \\\n    --hash=sha256:bf1f8a81d04ca90e32a0aceb819d34dbd378a98bf923b6398b9a3ec0bf44de29 \\\n    --hash=sha256:c46467899aaa4da076d5abc11084634e2d197e9460643dd455ac3db5856b24d6 \\\n    --hash=sha256:c4fc4c21971a1a9f4bdb4c73978c7f7256caa3e62b323f70d6cb80db583350bc \\\n    --hash=sha256:c503ba5216814e295f40711470446bc3fd00f0faea8a086cbc688808e26f92a2 \\\n    --hash=sha256:d051c0e065b94b7a3cea50eb1ec32e912cd96dba41647eb24104b6c6c14c5788 \\\n    --hash=sha256:d3e28b3e83862ccf4d85ff19cf8c20b2ae7e503881711ff2d534dc8f761131aa \\\n    --hash=sha256:db4301b2d1f926ae677a751eb2bd0e8c5f5319c9cb3f88b0becbbb0b07b34151 \\\n    --hash=sha256:dd7478f1463441ae4ca7308a70e90b33470fa593429f9d4c578dd00d1fa78838 \\\n    --hash=sha256:e05e1af93b977f7eafa636d043f9f94c7ee3ac81af99c13508215942e64c993b \\\n    --hash=sha256:e19d192383eab2f4ceb30b412b22ea30690c9e618f78870357ae1d682912015a \\\n    --hash=sha256:e32e7cc9af0f1cc15548288a51a3b681cc2a219faa838e995f7dc53dbab1062d \\\n    --hash=sha256:ecaf1e12bdc03c86ad4a7ea848d66c685cb6851d807a26aa245ca3d2017a1908 \\\n    --hash=sha256:ee15f284898e7b246df8087fc82b87b01686f98ee67d85a17b7ab44143a3a9a0 \\\n    --hash=sha256:ee67acbbf05014ea6c763beb097e03cd629961c8a632075eeb34247120abcb4b \\\n    --hash=sha256:f086f6fe114e19d92014a1966f43a3e62285109afe874f067f5abbdcbb10e59c \\\n    --hash=sha256:f8bfc0e12dc78f777f323f55c58649591b2cd0c43534e8355c51d3fede5f4dee\n    # via tirex-forecasting-pipeline (pyproject.toml)\npillow==12.3.0 \\\n    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \\\n    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \\\n    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \\\n    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \\\n    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \\\n    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \\\n    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \\\n    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \\\n    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \\\n    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \\\n    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \\\n    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \\\n    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \\\n    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \\\n    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \\\n    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \\\n    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \\\n    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \\\n    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \\\n    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \\\n    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \\\n    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \\\n    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \\\n    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \\\n    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \\\n    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \\\n    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \\\n    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \\\n    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \\\n    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \\\n    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \\\n    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \\\n    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \\\n    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \\\n    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \\\n    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \\\n    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \\\n    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \\\n    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \\\n    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \\\n    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \\\n    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \\\n    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \\\n    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \\\n    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \\\n    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \\\n    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \\\n    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \\\n    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \\\n    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \\\n    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \\\n    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \\\n    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \\\n    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \\\n    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \\\n    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \\\n    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \\\n    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \\\n    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \\\n    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \\\n    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \\\n    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \\\n    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \\\n    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \\\n    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \\\n    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \\\n    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \\\n    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \\\n    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \\\n    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \\\n    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \\\n    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \\\n    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \\\n    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \\\n    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \\\n    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \\\n    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \\\n    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \\\n    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \\\n    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \\\n    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \\\n    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \\\n    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \\\n    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \\\n    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \\\n    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \\\n    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7\n    # via\n    #   matplotlib\n    #   torchvision\npyparsing==3.3.3 \\\n    --hash=sha256:928ae7e20211f3b6f3915a72f06a0cfd29ab9d24279dd6346b6b1a7146397d36 \\\n    --hash=sha256:ece8c00a69cf01b45d0b1dedabb469c90d8caf996d4fda40f147627a122849a4\n    # via matplotlib\npython-dateutil==2.9.0.post0 \\\n    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \\\n    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427\n    # via\n    #   matplotlib\n    #   pandas\npytz==2026.5 \\\n    --hash=sha256:e658af3757f9e26a9d25dd2aff38335acd92bc9104f890a894b2c1ba28311b03 \\\n    --hash=sha256:fa23724b9c486543b9ff54a327ee7569ac83ade54bb9afd0fc18676620401c86\n    # via pandas\npyyaml==6.0.3 \\\n    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n    # via\n    #   huggingface-hub\n    #   mlstm-kernels\n    #   tirex-2\n    #   xlstm\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via huggingface-hub\nsafetensors==0.8.0 \\\n    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n    # via xlstm\nsetuptools==84.0.0 \\\n    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n    # via\n    #   torch\n    #   triton\nsix==1.17.0 \\\n    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \\\n    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81\n    # via python-dateutil\nsympy==1.14.0 \\\n    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n    # via torch\ntirex-2==0.2.1 \\\n    --hash=sha256:e8f978f5e40147c8a1df210e74b1058810edddafc96d9c120ba05ef75f5869e4 \\\n    --hash=sha256:f814f79cfab0ce87c781514e5afe329b78a314b727ae3349edbd34f602681f92\n    # via tirex-forecasting-pipeline (pyproject.toml)\ntorch==2.8.0 \\\n    --hash=sha256:06fcee8000e5c62a9f3e52a688b9c5abb7c6228d0e56e3452983416025c41381 \\\n    --hash=sha256:0be92c08b44009d4131d1ff7a8060d10bafdb7ddcb7359ef8d8c5169007ea905 \\\n    --hash=sha256:1a62a1ec4b0498930e2543535cf70b1bef8c777713de7ceb84cd79115f553767 \\\n    --hash=sha256:220a06fd7af8b653c35d359dfe1aaf32f65aa85befa342629f716acb134b9710 \\\n    --hash=sha256:2b2f96814e0345f5a5aed9bf9734efa913678ed19caf6dc2cddb7930672d6128 \\\n    --hash=sha256:2f4ac52f0130275d7517b03a33d2493bab3693c83dcfadf4f81688ea82147d2e \\\n    --hash=sha256:5128fe752a355d9308e56af1ad28b15266fe2da5948660fad44de9e3a9e36e8c \\\n    --hash=sha256:5ae0524688fb6707c57a530c2325e13bb0090b745ba7b4a2cd6a3ce262572916 \\\n    --hash=sha256:619c2869db3ada2c0105487ba21b5008defcc472d23f8b80ed91ac4a380283b0 \\\n    --hash=sha256:65616ca8ec6f43245e1f5f296603e33923f4c30f93d65e103d9e50c25b35150b \\\n    --hash=sha256:659df54119ae03e83a800addc125856effda88b016dfc54d9f65215c3975be16 \\\n    --hash=sha256:7b677e17f5a3e69fdef7eb3b9da72622f8d322692930297e4ccb52fefc6c8211 \\\n    --hash=sha256:83c13411a26fac3d101fe8035a6b0476ae606deb8688e904e796a3534c197def \\\n    --hash=sha256:89aa9ee820bb39d4d72b794345cccef106b574508dd17dbec457949678c76011 \\\n    --hash=sha256:8c7ef765e27551b2fbfc0f41bcf270e1292d9bf79f8e0724848b1682be6e80aa \\\n    --hash=sha256:8f0a9d617a66509ded240add3754e462430a6c1fc5589f86c17b433dd808f97a \\\n    --hash=sha256:a3f16a58a9a800f589b26d47ee15aca3acf065546137fc2af039876135f4c760 \\\n    --hash=sha256:a7242b86f42be98ac674b88a4988643b9bc6145437ec8f048fea23f72feb5eca \\\n    --hash=sha256:b2aca0939fb7e4d842561febbd4ffda67a8e958ff725c1c27e244e85e982173c \\\n    --hash=sha256:c12fa219f51a933d5f80eeb3a7a5d0cbe9168c0a14bbb4055f1979431660879b \\\n    --hash=sha256:da6afa31c13b669d4ba49d8a2169f0db2c3ec6bec4af898aa714f401d4c38904 \\\n    --hash=sha256:e2fab4153768d433f8ed9279c8133a114a034a61e77a3a104dcdf54388838705 \\\n    --hash=sha256:e8e5bf982e87e2b59d932769938b698858c64cc53753894be25629bdf5cf2f46 \\\n    --hash=sha256:e9f071f5b52a9f6970dc8a919694b27a91ae9dc08898b2b988abbef5eddfd1ae\n    # via\n    #   tirex-forecasting-pipeline (pyproject.toml)\n    #   flashrnn\n    #   mlstm-kernels\n    #   tirex-2\n    #   torchaudio\n    #   torchvision\n    #   xlstm\ntorchaudio==2.8.0 \\\n    --hash=sha256:078105bf80f725c0215a0bebac8cb2fb1b3993ab32bdc3fcd50145a5b4127001 \\\n    --hash=sha256:09535a9b727c0793cd07c1ace99f3f353626281bcc3e30c2f2314e3ebc9d3f96 \\\n    --hash=sha256:1951f10ed092f2dda57634f6a3950ef21c9d9352551aa84a9fccd51bbda18095 \\\n    --hash=sha256:4573c6042950c20278e3608a9a38050ba0bc72e0049e1bbfd249caf859a8029b \\\n    --hash=sha256:4b82cacd1b8ccd543b1149d8cab257a40dfda8119023d2e3a96c66349c84bffb \\\n    --hash=sha256:4e2b4712ad6d7547ce82d84567c8c29d5e2966ff1d31d94e1644024fb4b2649f \\\n    --hash=sha256:4f7d97494698d98854129349b12061e8c3398d33bd84c929fa9aed5fd1389f73 \\\n    --hash=sha256:522289e2cd57e79401fd5ccae9b1bc0ff2e47f3529092adf5acf57427ea0c6a9 \\\n    --hash=sha256:58f912bf2d289c709b42a55475b2b483becec79d9affb7684b606bb1f896b434 \\\n    --hash=sha256:68df9c9068984edff8065c2b6656725e6114fe89281b0cf122c7505305fc98a4 \\\n    --hash=sha256:776c0b4ba84b9e3ddf6304b9c47cd63549d7896a6f3d5184ece074cc3d76ed6b \\\n    --hash=sha256:862e2e40bf09d865e5df080a84c1a39bbcef40e43140f4b1737eb3a389d3b38f \\\n    --hash=sha256:93a8583f280fe83ba021aa713319381ea71362cc87b67ee38e97a43cb2254aee \\\n    --hash=sha256:a1f4bde9ce9316b6b24304c73ea9f82a2aabc4dbf8c6c2598117ea7c6ecc4db2 \\\n    --hash=sha256:c1b5139c840367a7855a062a06688a416619f6fd2ca46d9b9299b49a7d133dfd \\\n    --hash=sha256:c2f44cf279f673cfcdd8f576c349eee8bedf8caab351a5dd78b32970cc34a212 \\\n    --hash=sha256:c9276857d241c6de257af765c0f51fc011af38cb725401495121b280913007cf \\\n    --hash=sha256:d2a85b124494736241884372fe1c6dd8c15e9bc1931bd325838c5c00238c7378 \\\n    --hash=sha256:d3c1b85b26a09832d139f6d6da6b66caeb51d2e16e08f8587665c44a9e1aa8f9 \\\n    --hash=sha256:d4a715d09ac28c920d031ee1e60ecbc91e8a5079ad8c61c0277e658436c821a6 \\\n    --hash=sha256:ddef94bf181e6447cbb05f38beaca8f6c5bb8d2b9ddced1aa3452025b9fc70d3 \\\n    --hash=sha256:e30b4741ec333dea78d012047c58305a1865a20ce74a3b8bc2d7b40de344dd02 \\\n    --hash=sha256:ef7f7ffa828b8d8ba5d3a569b825fc0469688e1e8962bf6577d538bd8af1387d \\\n    --hash=sha256:f851d32e94ca05e470f0c60e25726ec1e0eb71cb2ca5a0206b7fd03272ccc3c8\n    # via tirex-forecasting-pipeline (pyproject.toml)\ntorchvision==0.23.0 \\\n    --hash=sha256:01dc33ee24c79148aee7cdbcf34ae8a3c9da1674a591e781577b716d233b1fa6 \\\n    --hash=sha256:07d069cb29691ff566e3b7f11f20d91044f079e1dbdc9d72e0655899a9b06938 \\\n    --hash=sha256:09bfde260e7963a15b80c9e442faa9f021c7e7f877ac0a36ca6561b367185013 \\\n    --hash=sha256:1c37e325e09a184b730c3ef51424f383ec5745378dc0eca244520aca29722600 \\\n    --hash=sha256:2a3299d2b1d5a7aed2d3b6ffb69c672ca8830671967eb1cee1497bacd82fe47b \\\n    --hash=sha256:2df618e1143805a7673aaf82cb5720dd9112d4e771983156aaf2ffff692eebf9 \\\n    --hash=sha256:2f7fd6c15f3697e80627b77934f77705f3bc0e98278b989b2655de01f6903e1d \\\n    --hash=sha256:31c583ba27426a3a04eca8c05450524105c1564db41be6632f7536ef405a6de2 \\\n    --hash=sha256:35c27941831b653f5101edfe62c03d196c13f32139310519e8228f35eae0e96a \\\n    --hash=sha256:3932bf67256f2d095ce90a9f826f6033694c818856f4bb26794cf2ce64253e53 \\\n    --hash=sha256:49aa20e21f0c2bd458c71d7b449776cbd5f16693dd5807195a820612b8a229b7 \\\n    --hash=sha256:4e7d31c43bc7cbecbb1a5652ac0106b436aa66e26437585fc2c4b2cf04d6014c \\\n    --hash=sha256:6c74cbc1cbee26dd4f35f989cd80dccc40411f258dee476b29871dee4b483af0 \\\n    --hash=sha256:6dd7c4d329a0e03157803031bc856220c6155ef08c26d4f5bbac938acecf0948 \\\n    --hash=sha256:7266871daca00ad46d1c073e55d972179d12a58fa5c9adec9a3db9bbed71284a \\\n    --hash=sha256:76bc4c0b63d5114aa81281390f8472a12a6a35ce9906e67ea6044e5af4cab60c \\\n    --hash=sha256:83ee5bf827d61a8af14620c0a61d8608558638ac9c3bac8adb7b27138e2147d1 \\\n    --hash=sha256:a2e45272abe7b8bf0d06c405e78521b5757be1bd0ed7e5cd78120f7fdd4cbf35 \\\n    --hash=sha256:a76fafe113b2977be3a21bf78f115438c1f88631d7a87203acb3dd6ae55889e6 \\\n    --hash=sha256:a9e9d7552d34547b80843eaf64ab0737b19b2e8bec2514286b8cfd30861ca8b5 \\\n    --hash=sha256:b190db205f90206c230fc2f91cbdfd5733334babc0e0d19bddb90a40b8cf26c2 \\\n    --hash=sha256:b9e2dabf0da9c8aa9ea241afb63a8f3e98489e706b22ac3f30416a1be377153b \\\n    --hash=sha256:dc7ce5accbbb8c9df9a79f8cef6a6df042f28e2250a6ae0d2ca70b06473fa03b \\\n    --hash=sha256:e0e2c04a91403e8dd3af9756c6a024a1d9c0ed9c0d592a8314ded8f4fe30d440\n    # via tirex-forecasting-pipeline (pyproject.toml)\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via huggingface-hub\ntriton==3.4.0 \\\n    --hash=sha256:00be2964616f4c619193cb0d1b29a99bd4b001d7dc333816073f92cf2a8ccdeb \\\n    --hash=sha256:31c1d84a5c0ec2c0f8e8a072d7fd150cab84a9c239eaddc6706c081bfae4eb04 \\\n    --hash=sha256:7936b18a3499ed62059414d7df563e6c163c5e16c3773678a3ee3d417865035d \\\n    --hash=sha256:7b70f5e6a41e52e48cfc087436c8a28c17ff98db369447bcaff3b887a3ab4467 \\\n    --hash=sha256:7ff2785de9bc02f500e085420273bb5cc9c9bb767584a4aa28d6e360cec70128 \\\n    --hash=sha256:98e5c1442eaeabae2e2452ae765801bd53cd4ce873cab0d1bdd59a32ab2d9397\n    # via\n    #   flashrnn\n    #   torch\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   huggingface-hub\n    #   torch\ntzdata==2026.5 \\\n    --hash=sha256:8cc73c0a0bfca7dbfa59235d60b2eff82231dee33f53d206db1acd9173cfc0a7 \\\n    --hash=sha256:b683bd1b6659ddcd810ff02ad09ba821d4bf1065072805063eb35c49617905ac\n    # via pandas\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via requests\nxlstm==2.0.6 \\\n    --hash=sha256:1544dfec22a96dfb92464912e2caa9ac4b92b9ecd2be8cb2e4dfb22373b2b1eb \\\n    --hash=sha256:8fcb24b45a447fa255bbe4b4306aa93e6d4bd7cf06a6cd08d9494986662a141a\n    # via tirex-2\n', 'weights/tirex-2/dimer-base-manifest.json': '{\n  "format": "dimer_hf_snapshot",\n  "formatVersion": 1,\n  "modelKey": "tirex-2",\n  "modelId": "NX-AI/TiRex-2",\n  "revision": "05e5b26db52bfb256f1ae1bdf785589850482de3",\n  "files": [\n    {\n      "path": "README.md",\n      "bytes": 5521,\n      "sha256": "ec10f277ff820fb8915b4ff3988916c25bc892167902aba84bb9bfe24d190b70"\n    },\n    {\n      "path": "model-config.yaml",\n      "bytes": 1204,\n      "sha256": "cddbe6d0be4919cb7b0cad747fb19a295264e129d1c0790cf62133b4cb5da727"\n    },\n    {\n      "path": "model.ckpt",\n      "bytes": 380613375,\n      "sha256": "184b160ffbe4c01a26beeba14015ff3507c7497e1f3577114187bbc1d19fcac1"\n    }\n  ],\n  "totalBytes": 380620100\n}\n', 'LICENSE': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or Derivative\n          Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2026 Kurt Valcorza\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'source.json': '{\n  "repository": "kurtvalcorza/tirex-forecasting-pipeline",\n  "revision": "1df20d15d9128befbf3671e45fbc4f904b242398",\n  "generator": "build_notebook.py/3.0",\n  "notebook_spec": "2.2",\n  "files": {\n    "src/tirex_forecasting_pipeline/__init__.py": "022190fb0c8bbbf7f2d9fc249c04065de277bb95cbfd2faf11cbd5fce018c249",\n    "src/tirex_forecasting_pipeline/evaluation.py": "a0d38e5a8426501b424c31aedff5308b9d9a7f90745d4b0c26fa6f86ded82bd4",\n    "src/tirex_forecasting_pipeline/pipeline.py": "f2ed9d8c2663380c2b93427301b80af878102175634debfb79a350076dd33b58",\n    "src/tirex_forecasting_pipeline/validation.py": "29ad4a59f789edc7c3a740d837b9e58581830ee7cddb4b2d1bafbded7af52ad5",\n    "tutorial_stages.py": "a280bce0f65e093952e675fdb97612fe5ca4b89142df94c6eadce1a206a508d3",\n    "requirements.txt": "f3848bd09caed0ba8e40e94c10efbdbad91c1bb14f8008f7ef3ff0a1af03fd3f",\n    "weights/tirex-2/dimer-base-manifest.json": "07e2fe2da6a0a79a2349740472df4c64a7a3263b6b3bce9843cd890230293727",\n    "LICENSE": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138"\n  },\n  "sources": {\n    "src/tirex_forecasting_pipeline/__init__.py": "src/tirex_forecasting_pipeline/__init__.py",\n    "src/tirex_forecasting_pipeline/evaluation.py": "src/tirex_forecasting_pipeline/evaluation.py",\n    "src/tirex_forecasting_pipeline/pipeline.py": "src/tirex_forecasting_pipeline/pipeline.py",\n    "src/tirex_forecasting_pipeline/validation.py": "src/tirex_forecasting_pipeline/validation.py",\n    "tutorial_stages.py": "tools/tutorial_stages.py",\n    "requirements.txt": "tutorials/requirements-colab.lock.txt",\n    "weights/tirex-2/dimer-base-manifest.json": "weights/tirex-2/dimer-base-manifest.json",\n    "LICENSE": "LICENSE"\n  }\n}\n'}
CARRIED_BINARY = {}
CARRIED_HASHES = {'src/tirex_forecasting_pipeline/__init__.py': '022190fb0c8bbbf7f2d9fc249c04065de277bb95cbfd2faf11cbd5fce018c249', 'src/tirex_forecasting_pipeline/evaluation.py': 'a0d38e5a8426501b424c31aedff5308b9d9a7f90745d4b0c26fa6f86ded82bd4', 'src/tirex_forecasting_pipeline/pipeline.py': 'f2ed9d8c2663380c2b93427301b80af878102175634debfb79a350076dd33b58', 'src/tirex_forecasting_pipeline/validation.py': '29ad4a59f789edc7c3a740d837b9e58581830ee7cddb4b2d1bafbded7af52ad5', 'tutorial_stages.py': 'a280bce0f65e093952e675fdb97612fe5ca4b89142df94c6eadce1a206a508d3', 'requirements.txt': 'f3848bd09caed0ba8e40e94c10efbdbad91c1bb14f8008f7ef3ff0a1af03fd3f', 'weights/tirex-2/dimer-base-manifest.json': '07e2fe2da6a0a79a2349740472df4c64a7a3263b6b3bce9843cd890230293727', 'LICENSE': 'f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138', 'source.json': '3418b67a7e4667014efc9f39cf5a565a5c8982f8edb539da0c41a02e973efed7'}
for name, payload in [*((n, t.encode('utf-8')) for n, t in CARRIED_FILES.items()), *((n, base64.b64decode(b)) for n, b in CARRIED_BINARY.items())]:
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    if hashlib.sha256(path.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried file integrity failure: ' + name + '. Do not edit this cell; regenerate the notebook from the repository.')
NOTEBOOK_SOURCE = json.loads((ROOT / 'source.json').read_text(encoding='utf-8'))
print({'carried_files': len(CARRIED_HASHES), 'verified': True, 'repository': NOTEBOOK_SOURCE['repository'], 'revision': NOTEBOOK_SOURCE['revision'], 'generator': NOTEBOOK_SOURCE['generator']})

{'carried_files': 9, 'verified': True, 'repository': 'kurtvalcorza/tirex-forecasting-pipeline', 'revision': '1df20d15d9128befbf3671e45fbc4f904b242398', 'generator': 'build_notebook.py/3.0'}


**Infrastructure: the isolated environment.** This cell installs **nothing into the notebook kernel**. It downloads one pinned file — the `uv` 0.12.15 installer wheel, refused unless its size and SHA-256 match — creates a separate virtual environment with its own CPython 3.12.12, and installs `requirements.txt` into it with `--require-hashes`: every package must be the locked version and match a locked digest. The hosted runtime's own packages (Colab preloads its own NumPy, pandas and PyTorch) are never replaced, which is why no restart is needed. The environment is keyed on the lock's SHA-256: a second **Run all** in the same runtime reuses it and prints `environment_reused: True`. The first build downloads PyTorch with its CUDA libraries (a few GB) and typically takes a few minutes. The cell also defines `run_stage` and `load_record`, the two helpers the learner cells use: each learner cell runs one stage of the carried runner in its own process, streams what it prints, and stops with the stage's own error message if it fails.

**Expected result:** one dictionary with the generating revision, the isolated environment's Python, the `torch`, `tirex-2`, `numpy`, `pandas` versions, whether CUDA is visible, the number of locked packages, whether an existing environment was reused, and the setup time. A failed download or a hash mismatch stops the cell; never remove a pin or a hash to get past one.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated locked environment and define the stage runner
import io
import urllib.error
import urllib.request
import zipfile

LOCK_SHA256 = 'f3848bd09caed0ba8e40e94c10efbdbad91c1bb14f8008f7ef3ff0a1af03fd3f'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
ENV_ROOT.mkdir(parents=True, exist_ok=True)
UV = ENV_ROOT / 'uv-0.12.15'
UV_MARK = ENV_ROOT / 'uv-0.12.15.sha256'  # digest of the binary extracted from the verified wheel
if not (UV.is_file() and UV_MARK.is_file() and hashlib.sha256(UV.read_bytes()).hexdigest() == UV_MARK.read_text().strip()):
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2 ** (attempt + 1))
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('uv 0.12.15 wheel size/hash mismatch: refusing to run it')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
        UV.write_bytes(archive.read(member))
    UV.chmod(0o700)
    UV_MARK.write_text(hashlib.sha256(UV.read_bytes()).hexdigest())
# Stage processes inherit no kernel Python path or start-up file and no credentials (every download is public). The
# kernel may export an inline matplotlib backend that the isolated environment cannot import; stages write files.
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1', UV_CACHE_DIR=str(ENV_ROOT / 'cache'), MPLBACKEND='Agg', PYTHONNOUSERSITE='1')
for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'GOOGLE_APPLICATION_CREDENTIALS'):
    ENV.pop(name, None)
# One environment per lock digest: a second Run all (or a re-run of this cell) reuses a complete matching environment.
VENV = ENV_ROOT / ('venv-' + LOCK_SHA256[:16])
READY = VENV / '.dimer-ready'
PYTHON = VENV / 'bin' / 'python'
environment_reused = READY.is_file() and READY.read_text().strip() == LOCK_SHA256 and PYTHON.is_file()
if not environment_reused:
    shutil.rmtree(VENV, ignore_errors=True)
    subprocess.run([str(UV), 'venv', '--managed-python', '--python', '3.12.12', str(VENV)], env=ENV, check=True)
    subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '--require-hashes', *['--only-binary', ':all:'], '--index-url', 'https://pypi.org/simple', '-r', str(ROOT / 'requirements.txt')], env=ENV, check=True)
    READY.write_text(LOCK_SHA256)
probe = subprocess.run([str(PYTHON), '-c', "import importlib.metadata, json, platform, torch; print(json.dumps({'python': platform.python_version(), **{d: importlib.metadata.version(d) for d in ['torch', 'tirex-2', 'numpy', 'pandas']}, 'cuda': torch.cuda.is_available()}))"], env=ENV, check=True, capture_output=True, text=True)
RUNTIME = json.loads(probe.stdout.strip().splitlines()[-1])
print({'notebook_source': NOTEBOOK_SOURCE['revision'], **RUNTIME, 'locked_packages': 57, 'environment': str(VENV), 'environment_reused': environment_reused, 'setup_seconds': round(time.perf_counter() - SESSION_START)})
if not RUNTIME['cuda']:
    print('The isolated environment sees no CUDA GPU; this notebook runs its model on the CPU in any case (the reference path).')


def run_stage(stage, **options):
    """Run one stage of the carried runner in its own process with the isolated interpreter; stream its output."""
    if not (ROOT / 'tutorial_stages.py').is_file() or not PYTHON.is_file():
        raise RuntimeError(f'The run directory {ROOT} has no carried files, or the isolated environment is gone: run the Infrastructure cells again in order (Sections 1, 2 and 3), or choose Runtime > Run all.')
    log = ROOT / 'logs' / (stage + '.log')
    log.parent.mkdir(exist_ok=True)
    error_file = ROOT / 'state' / (stage + '.error.json')
    error_file.unlink(missing_ok=True)
    command = [str(PYTHON), '-u', str(ROOT / 'tutorial_stages.py'), '--root', str(ROOT), '--outputs', str(OUTPUTS), '--weights', str(WEIGHTS), '--stage', stage, '--options', json.dumps(options)]
    print('Running stage', repr(stage), 'in the isolated environment; log:', log, flush=True)
    with log.open('w', encoding='utf-8') as output:
        process = subprocess.Popen(command, env=ENV, cwd=str(ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            output.write(line)
            output.flush()
            if line.strip() and len(line) < 4000:
                print(line.rstrip(), flush=True)
        process.wait()
    if process.returncode:
        detail = 'see the log above'
        if error_file.is_file():
            error = json.loads(error_file.read_text(encoding='utf-8'))
            detail = error['type'] + ': ' + error['message']
        raise RuntimeError(f'Stage {stage!r} failed (exit {process.returncode}): {detail}')


def load_record(name):
    """A JSON record a stage wrote to OUTPUTS."""
    return json.loads((OUTPUTS / name).read_text(encoding='utf-8'))

{'notebook_source': '1df20d15d9128befbf3671e45fbc4f904b242398', 'python': '3.12.12', 'torch': '2.8.0', 'tirex-2': '0.2.1', 'numpy': '2.3.3', 'pandas': '2.3.3', 'cuda': True, 'locked_packages': 57, 'environment': '/tmp/dimer_uv/venv-f3848bd09caed0ba', 'environment_reused': False, 'setup_seconds': 79}


## 3. Pin, stage and verify the model · [Engineering]

> **Infrastructure.** The next code cell is collapsed. It downloads one file and checks it; you may run it without studying its implementation.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the carried package and the 3-file manifest (path, byte size, SHA-256) — and the `weights` stage first checks that they agree. `stage_missing_files(..., allow_download=True)` then fetches exactly the absent entries from the Hugging Face Hub **at revision `05e5b26db52b…`** (never `main`, ~381 MB), and `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch. There is no fallback to a different download and no remote model code is executed.

**What to notice:** the model id, revision, licence and file count; `fetched` lists the checkpoint on a first run and is empty on a rerun (staging only fetches a missing file); then `verified_files`. A mismatch stops the cell with an error naming the file — see **Troubleshooting**, and never edit a manifest to get past one.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned checkpoint
run_stage('weights')

Running stage 'weights' in the isolated environment; log: /content/dimer_runs/tirex_forecasting/7bd860dc1dce/logs/weights.log


{'model_id': 'NX-AI/TiRex-2', 'revision': '05e5b26db52bfb256f1ae1bdf785589850482de3', 'license': 'Apache-2.0', 'files': 3, 'total_bytes': 380620100}


{'weights_dir': '/content/weights/tirex-2', 'fetched': ['README.md', 'model-config.yaml', 'model.ckpt']}


{'verified': {'path': '/content/weights/tirex-2', 'format': 'dimer_hf_snapshot', 'formatVersion': 1, 'modelKey': 'tirex-2', 'modelId': 'NX-AI/TiRex-2', 'revision': '05e5b26db52bfb256f1ae1bdf785589850482de3', 'files': [{'path': 'README.md', 'bytes': 5521, 'sha256': 'ec10f277ff820fb8915b4ff3988916c25bc892167902aba84bb9bfe24d190b70'}, {'path': 'model-config.yaml', 'bytes': 1204, 'sha256': 'cddbe6d0be4919cb7b0cad747fb19a295264e129d1c0790cf62133b4cb5da727'}, {'path': 'model.ckpt', 'bytes': 380613375, 'sha256': '184b160ffbe4c01a26beeba14015ff3507c7497e1f3577114187bbc1d19fcac1'}], 'totalBytes': 380620100}, 'checkpoint': 'model.ckpt', 'note': 'model.ckpt is a PyTorch checkpoint (a pickle container); tirex2 loads it with weights_only, and it is accepted only at the pinned size and SHA-256'}


**Trust boundary (Section 3).** The checkpoint is `model.ckpt`, a PyTorch checkpoint — a pickle container — because no safetensors file is published at the pinned revision. It is accepted only at the pinned size and SHA-256, and `tirex2` loads it with `weights_only`, which refuses arbitrary pickled objects. **What to notice:** `fetched` lists `model.ckpt` on a first run, then the verified-file count.

## 4. Generate the synthetic sample or supply your own series · [Evaluation practice]

The default sample is **synthetic**: 256 steps of a linear trend plus a sine season plus small Gaussian noise from a fixed seed, so it needs no download and its float32 SHA-256 is printed for the record. It has a real future — the final `HORIZON` steps are withheld in the next section — so the evaluation can score the forecast, but a synthetic series says nothing about any deployment domain.

**Bring your own series.** Set `USE_BYOD = True` and `BYOD_CSV_PATH` (or, in Colab, leave it empty for the upload dialog). The header is inspected before pandas reads the file, so duplicate names cannot be silently renamed; the delimiter is detected (comma, semicolon or tab); a text value in a target column is refused with the column and the values; empty values, unparseable or irregular timestamps and too few rows stop with their own message; and a series longer than `MAX_CONTEXT + HORIZON` keeps its most recent rows and reports the trim.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_CSV_PATH = ''  # @param {type:"string"}
HORIZON = 32  # @param {type:"integer"}

def upload_one(what, field):
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError(f'{field} is empty, and the upload dialog exists only in Google Colab: set {field} to {what} in this runtime.') from None
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f'The upload was cancelled or empty: no file was received. Run this cell again and choose {what}, or set {field}.')
    if len(uploaded) != 1:
        raise ValueError(f'Upload exactly one file ({what}); got {sorted(uploaded)}.')
    upload_name, payload = next(iter(uploaded.items()))
    path = ROOT / 'inputs' / Path(upload_name).name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    return str(path)

csv_path = ''
if USE_BYOD:
    csv_path = BYOD_CSV_PATH or upload_one('one CSV with a timestamp column', 'BYOD_CSV_PATH')
run_stage('data', use_byod=USE_BYOD, byod_csv_path=csv_path, horizon=HORIZON)

Running stage 'data' in the isolated environment; log: /content/dimer_runs/tirex_forecasting/7bd860dc1dce/logs/data.log


{'sample_kind': 'synthetic', 'name': 'synthetic_trend_season_256.csv', 'generator': '0.02·t + sin(t/8) + N(0, 0.05), seed 7', 'shape': [1, 256], 'horizon': 32, 'float32_sha256': '55e436de917b774696580863e4c474e7444bf0747db219e724711b46e10dbeee'}


**What to notice:** `sample_kind: 'synthetic'`, shape `[1, 256]`, `horizon: 32` and the float32 SHA-256 `55e436de…`.

## 5. Chronological holdout, validation and baselines · [Evaluation practice]

The final `HORIZON` steps are **withheld** as the truth; only the earlier context is passed to the model, so no future value leaks into the forecast. `validate_inputs` applies exactly the checks `forecast` applies — target shape, context length `MIN_CONTEXT`..`MAX_CONTEXT`, horizon 1..`MAX_HORIZON`, finiteness and the covariate time-axis contract — and returns an **input manifest**, written to `outputs/tirex_forecasting_input_manifest.json`; the stage also validates a deliberately too-short context and records the refusal as a finding.

Two baselines are computed from the same context: the **last-value baseline**, and a **least-squares trend + season reference** — level, slope and one sine/cosine pair fitted to the context. For the synthetic series its period is the generator's own (2π·8 ≈ 50.3 steps): the right functional family, which is what makes it a fair reference. For your own series set `SEASON_PERIOD` in steps (for example 7 for daily data with a weekly cycle), or leave it 0 to use the largest periodogram peak of the detrended context. For the synthetic series the stage also prints the **noise floor**: the error of knowing the signal exactly.

In [ ]:
SEASON_PERIOD = 0.0  # @param {type:"number"}
run_stage('validate', season_period=SEASON_PERIOD)

Running stage 'validate' in the isolated environment; log: /content/dimer_runs/tirex_forecasting/7bd860dc1dce/logs/validate.log


{'ceilings': {'MIN_CONTEXT': 32, 'MAX_CONTEXT': 16384, 'MAX_HORIZON': 4096}}


{


  "schema": {


    "input": "1D array (time) or 2D array (variates, time) of finite numbers, cast to float32",


...

  "model_revision": "05e5b26db52bfb256f1ae1bdf785589850482de3"


}


{'last_value_baseline': {'mae': 0.6729, 'rmse': 0.7468}}


{'least_squares_trend_season': {'mae': 0.0437, 'rmse': 0.0576, 'period_steps': 50.2655}}


{'noise_floor': {'mae': 0.0414, 'rmse': 0.0555}}


**What to notice:** context length 224, the `short-context-probe` refusal, last-value MAE 0.6729 / RMSE 0.7468, the trend + season reference 0.0437 / 0.0576, and the noise floor 0.0414 / 0.0555.

**Checkpoint:** why is the trend + season reference fitted on the context only, never on the withheld steps?

<details>
<summary>Check your reasoning (open after answering)</summary>

Fitting on the withheld steps would let the reference see the answer, exactly the leakage the chronological holdout exists to prevent. A baseline must play by the same rule as the model: the past only. That is also why its error is a fair reference point for the model's.

</details>

## 6. Forecast · [Concept]

`forecast` returns `quantiles` of shape `(variates, 9, horizon)` at levels 0.1 … 0.9, and `median` — the q=0.5 slice, which is the **point forecast**. The median is a model median, not a mean, and the other quantiles are model quantiles rather than guaranteed confidence intervals; nothing is calibrated here. Inference is zero-shot and runs on the CPU; it is deterministic given the same weights and library versions.

**Predict before running:** on this smooth series, will the first forecast steps sit close to the truth? Will the q10–q90 band widen or narrow further into the horizon?

In [ ]:
run_stage('forecast')

Running stage 'forecast' in the isolated environment; log: /content/dimer_runs/tirex_forecasting/7bd860dc1dce/logs/forecast.log


W1010 11:06:31.867000 1691 torch/utils/cpp_extension.py:118] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


W1010 11:06:32.020000 1691 torch/utils/cpp_extension.py:118] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


{'point_forecast': 'median (q=0.5)', 'horizon': 32, 'context_length': 224, 'n_variates': 1, 'device': 'cpu', 'source': 'local-snapshot', 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]}


step  1  median 4.6793  truth 4.7478  q10 4.5784  q90 4.8207


step  2  median 4.6027  truth 4.6691  q10 4.4691  q90 4.7520


step  3  median 4.5126  truth 4.4949  q10 4.3759  q90 4.6540


step  4  median 4.4134  truth 4.4066  q10 4.2591  q90 4.5654


step  5  median 4.3262  truth 4.2863  q10 4.1838  q90 4.4769


**What to notice:** `point_forecast: 'median (q=0.5)'`, `device: 'cpu'`, `source: 'local-snapshot'`, and the first steps of the median next to the truth, inside the q10–q90 band.

## 7. Evaluate → evaluation report · [Evaluation practice]

`evaluation_report` carries the repository's own `mae` and `rmse` on the median, the `interval_coverage` of the q10–q90 band (nominal 0.8), and the last-value baseline, with the verdict `sample-sanity` — one holdout, no dispersion estimate, not a benchmark (without withheld truth the verdict is `not-measurable`). The stage adds the least-squares trend + season reference, the noise floor (synthetic series), the coverage granularity (with 32 points, coverage moves in steps of 1/32 ≈ 0.031) and an **interpretation** line. The report is written to `outputs/tirex_forecasting_evaluation_report.json`.

**Predict before running:** TiRex's median versus the trend + season reference — which will have the lower MAE on this series, and by how much?

In [ ]:
run_stage('evaluate')

Running stage 'evaluate' in the isolated environment; log: /content/dimer_runs/tirex_forecasting/7bd860dc1dce/logs/evaluate.log


{


  "task": "zero-shot probabilistic time-series forecasting",


  "point_forecast": "median (q=0.5)",


  "score_semantics": "quantile levels 0.1..0.9 are model quantiles, not calibrated confidence intervals",


...

    }


  ],


  "coverage_granularity": "q10\u2013q90 coverage over 32 holdout point(s) moves in steps of 1/32 = 0.0312",


  "interpretation": "TiRex median MAE 0.0491 vs last-value 0.6729 and least-squares trend + season 0.0437; noise floor 0.0414: on this synthetic series a 4-parameter fit of the right family already sits near the floor, so beating last-value says little about model skill"


}


**What to notice:** the model's MAE (0.0491 recorded) beside last-value (0.6729), the trend + season reference (0.0437) and the noise floor (0.0414), the coverage and its granularity, and the interpretation line.

**Checkpoint:** the model is about 14 times better than last-value. Is that evidence of skill?

<details>
<summary>Check your reasoning (open after answering)</summary>

Not on this series. Last-value ignores both the trend and the season, so any method that models them looks good against it. A 4-parameter least-squares fit of the right family reaches 0.0437 — slightly better than the model's recorded 0.0491 — and both are near the noise floor of 0.0414, so there is almost nothing left to win. TiRex's result is a good zero-shot forecast with no knowledge of the series' form; whether it adds skill is a question for real series, compared against a seasonal reference over several holdouts.

</details>

## 8. Export outputs and provenance · [Engineering]

The CSV keeps variate, step, median, truth, the last-value baseline, the trend + season reference and all nine quantiles aligned row by row. `outputs/tirex_forecasting_result.json` preserves the forecast summary, the evaluation report, the input manifest, the sample identity and digest (with any trim), the notebook's source, the model identifier, revision and licence, and the isolated environment's versions and device. No credentials are recorded. The stage only reads earlier results, so re-running it alone is safe.

In [ ]:
run_stage('export')

Running stage 'export' in the isolated environment; log: /content/dimer_runs/tirex_forecasting/7bd860dc1dce/logs/export.log


{'rows': 32, 'columns': ['variate', 'step', 'median', 'truth', 'last_value_baseline', 'least_squares_trend_season', 'q10', 'q20', 'q30', 'q40', 'q50', 'q60', 'q70', 'q80', 'q90'], 'runtime': {'python': '3.12.12', 'torch': '2.8.0', 'tirex-2': '0.2.1', 'numpy': '2.3.3', 'pandas': '2.3.3', 'device': 'cpu'}}


['tirex_forecasting_evaluation_report.json', 'tirex_forecasting_forecast.csv', 'tirex_forecasting_input_manifest.json', 'tirex_forecasting_result.json']


## 9. Optional activity: how much context does the forecast need? · [Concept]

**Predict → Change → Run → Observe → Explain.** **Predict:** with `ACTIVITY_CONTEXT = 64` instead of the full 224 steps, will the MAE rise? Will the q10–q90 band widen? **Change:** tick `RUN_ACTIVITY` and set `ACTIVITY_CONTEXT` (between 32 and 224). **Run** this cell. **Observe** the two rows — full and short context — for MAE, mean band width and coverage. **Explain** the difference in terms of how many seasonal cycles the model can see. The activity writes only to `outputs/activity/`.

In [ ]:
RUN_ACTIVITY = False  # @param {type:"boolean"}
ACTIVITY_CONTEXT = 64  # @param {type:"integer"}
if RUN_ACTIVITY:
    run_stage('activity', context_length=ACTIVITY_CONTEXT)
else:
    print('Optional activity skipped: tick RUN_ACTIVITY to run it. The canonical outputs are complete.')

Optional activity skipped: tick RUN_ACTIVITY to run it. The canonical outputs are complete.


**What to notice (if you ran it):** the `mae` and `mean_q10_q90_width` columns.

<details>
<summary>Check your reasoning (open after running)</summary>

No hosted run of the activity is recorded yet, so compare your rows. 64 steps cover just over one season (about 50 steps), so the model has less evidence about the period and the trend; expect a larger error and usually a wider band. A band that widens as evidence shrinks is the quantiles behaving sensibly — but it is still not a calibrated interval.

</details>

## Troubleshooting · [Engineering]

| Symptom | Likely cause | What to do |
|---|---|---|
| `This notebook needs a Linux x86_64 runtime` | macOS, Windows or ARM kernel | Use Colab, Kaggle or a Linux x86_64 Jupyter kernel. |
| `Not enough free disk` | a used runtime | Start a fresh runtime. |
| `uv … wheel size/hash mismatch` or a `--require-hashes` error | a corrupted or substituted download | Re-run Section 2; never remove a pin or a hash. |
| a size or SHA-256 mismatch in Section 3 | a file that is not the pinned one | Delete `weights/tirex-2/` and re-run Section 3; never edit the manifest. |
| `A CUDA device is visible but no CUDA toolkit was found` | a GPU visible to upstream xlstm | Should not occur: the model stages hide the GPU; report the versions from Section 2 if it does. |
| `InvalidCxxCompiler` or `No working C++ compiler found` in Section 6 | a runtime without `g++` (TorchInductor compiles the CPU kernels of `torch.compile` on the first forecast) | Use Colab or Kaggle, or install a C++ compiler in the runtime. |
| `Stage '…' failed …: … is missing: run the stage that writes it` | a cell run out of order | Run the notebook in order from Section 4 (or *Run all*). |
| `BYOD_CSV_PATH is empty, and the upload dialog exists only in Google Colab` | BYOD outside Colab with no path | Set `BYOD_CSV_PATH`. |
| `The upload was cancelled or empty` / `Upload exactly one file` | a cancelled or multi-file upload | Run the cell again and choose one CSV. |
| `<file>: no timestamp column in the header` | a missing or misnamed column, or another delimiter | Name the column `timestamp`; comma, semicolon and tab are read. |
| `<file>: target columns must be numeric; non-numeric values in {…}` | a text or identifier column | Remove the column or fix the values. |
| `<file>: empty values in {…}` | gaps in a target | Fill or drop those rows; this pipeline does not impute. |
| `<file>: timestamps must be regularly spaced` / `unique and strictly increasing` | gaps, duplicates or unsorted rows | Sort, de-duplicate and resample first. |
| `<file>: N rows; this tutorial needs at least …` | too short for context + holdout | Supply more rows or lower `HORIZON`. |
| `trimmed: {…}` (a note, not an error) | more than `MAX_CONTEXT + HORIZON` rows | The most recent rows are kept; older rows cannot be context. |

## Interpretation and limits

The forecast is zero-shot; no gradient training or fine-tuning occurs. q=0.5 is a model median, and the other quantiles are model quantiles rather than guaranteed confidence intervals — the reported q10–q90 coverage on one holdout of 32 points moves in steps of 0.031 and is not a calibration statement. On the default synthetic series a 4-parameter trend + season fit of the right family is about as accurate as the model and both sit near the noise floor, so the comparison shows the workflow, not model skill; the last-value baseline alone would overstate what the model adds. MAE/RMSE from one chronological holdout must be repeated over representative periods of a real deployment series, against a seasonal reference, before any conclusion. Regime changes, missing values (rejected by the contract), irregular sampling and horizons far beyond the context all degrade results in ways the pipeline does not detect. The pipeline provides no fine-tuning, online adaptation, anomaly detection or imputation capability.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned checkpoint, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## Conclusion · [Evaluation practice]

Write three sentences: what the chronological holdout guarantees; how the model's error compares with the last-value baseline, the trend + season reference and the noise floor on this series; and what you would need before trusting the model on a real series.

<details>
<summary>Sample conclusion (open after writing yours)</summary>

The holdout withheld the last 32 steps before the model saw anything, so no future value leaked into the forecast or the baselines. The model's median MAE (0.0491 recorded) is far below last-value (0.6729) but slightly above a 4-parameter trend + season fit (0.0437), and both are near the noise floor (0.0414), so on this easy series the model shows a sound zero-shot workflow rather than skill. Before trusting it on a real series I would compare it with a seasonal reference over several rolling holdouts from representative periods, and check its band coverage over many points.

</details>

**Next experiments:** enable `USE_BYOD` with a CSV from your own domain and compare the median's MAE with both baselines over several consecutive holdouts (lower `HORIZON`, or trim the CSV and re-run); set `SEASON_PERIOD` to your series' cycle; run the activity at 32, 64 and 128 steps.

## References

- Repository README: https://github.com/kurtvalcorza/tirex-forecasting-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/tirex-forecasting-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/tirex-forecasting-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/NX-AI/TiRex-2
- Upstream code: https://github.com/NX-AI/tirex-2
- Paper: https://arxiv.org/abs/2607.01204